# 06. Factor Modeling

El objetivo de este cuaderno es evaluar la capacidad predictiva de los factores cuantitativos sobre los retornos futuros a 21 días mediante distintos algoritmos de aprendizaje automático (Regresión Lineal, Random Forest, XGBoost y LightGBM). 

Para ello, se aplicará un esquema riguroso de validación *Walk-Forward* con periodo de *purging* que prevenga el *look-ahead bias*. El flujo compara el desempeño entre las transformaciones Z-Score y Percentile Rank para determinar el pipeline óptimo. Finalmente, se seleccionará el modelo ganador y se analizará la relevancia económica de sus factores mediante valores SHAP.

## 1. Imports y Configuración


### 1.1 Librerías

In [1]:
import sys
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import random
import warnings
import optuna

from pathlib import Path
from itertools import combinations
from pprint import pprint

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

from src.models.training import run_cpcv_training
from src.models.utils import print_metrics
from src.models.tuning import optimize_hyperparameters
from src.models.utils import extract_mean_metrics

from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

c:\Users\dppec\OneDrive\Escritorio\PROYECTOS DE PYTHON\quant-portfolio-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.2 Configuración del notebook (semillas, suppress warnings, estilo)

In [2]:
# 1. Set global random seed for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# 2. Suppress non-critical warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# 3. Configure default plotting style and visual parameters
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 10

# 4. Configure Optuna logging

optuna.logging.set_verbosity(optuna.logging.WARNING)

### 1.3 Parámetros globales y rutas

In [3]:
# Factors and target variable for modeling
FACTORS = [
    "momentum_12_1",
    "upside_volatility",
    "log10_amihud"
]

TARGET = "forward_return_21d"

# Temporal parameters
FORWARD_HORIZON = 21
PURGE_WINDOW = 21

# Directory paths
DATA_DIR = Path("../data")
PREPROCESSED_DIR = DATA_DIR / "preprocessed"
MODELS_DIR = Path("../models")

# Ensure directories exist
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Carga de Datos y Verificación de Integridad

### 2.1 Carga de datasets

In [4]:
df_final_rank = pd.read_parquet("../data/preprocessed/df_final_rank.parquet")
df_final_z = pd.read_parquet("../data/preprocessed/df_final_z.parquet")

### 2.2 Verificación de tipos, fechas y dimensiones

In [5]:
from src.models.utils import verify_dataset_integrity

verify_dataset_integrity(df_final_z, "df_final_z (Z-Score)")

=== df_final_z (Z-Score) Integrity Verification ===
Dataset Shape: 1,749,937 rows x 10 columns
MultiIndex Levels: ['date', 'ticker']
Period Coverage: 2011-01-03 to 2024-12-30
Unique Dates: 3,521 | Unique Assets: 497

Data Types Summary: All columns are numeric (10 float64)

Missing values per column:
  - momentum_12_1: 112,172 (6.41%)
  - upside_volatility: 97,043 (5.55%)
  - log10_amihud: 95,135 (5.44%)
  - momentum_12_1_win: 112,172 (6.41%)
  - upside_volatility_win: 97,043 (5.55%)
  - log10_amihud_win: 95,135 (5.44%)
  - momentum_12_1_win_z: 112,172 (6.41%)
  - upside_volatility_win_z: 97,043 (5.55%)
  - log10_amihud_win_z: 95,135 (5.44%)
  - forward_return_21d: 104,525 (5.97%)
--------------------------------------------------



In [6]:
verify_dataset_integrity(df_final_rank, "df_final_rank (Percentile Rank)")

=== df_final_rank (Percentile Rank) Integrity Verification ===
Dataset Shape: 1,749,937 rows x 10 columns
MultiIndex Levels: ['date', 'ticker']
Period Coverage: 2011-01-03 to 2024-12-30
Unique Dates: 3,521 | Unique Assets: 497

Data Types Summary: All columns are numeric (10 float64)

Missing values per column:
  - momentum_12_1: 112,172 (6.41%)
  - upside_volatility: 97,043 (5.55%)
  - log10_amihud: 95,135 (5.44%)
  - momentum_12_1_win: 112,172 (6.41%)
  - upside_volatility_win: 97,043 (5.55%)
  - log10_amihud_win: 95,135 (5.44%)
  - momentum_12_1_win_rank: 112,172 (6.41%)
  - upside_volatility_win_rank: 97,043 (5.55%)
  - log10_amihud_win_rank: 95,135 (5.44%)
  - forward_return_21d: 104,525 (5.97%)
--------------------------------------------------



In [7]:
# Exact structural alignment check
same_index = df_final_z.index.equals(df_final_rank.index)
same_num_cols = len(df_final_z.columns) == len(df_final_rank.columns)
same_target = TARGET in df_final_z.columns and TARGET in df_final_rank.columns

print("Structural Alignment Check:")
print(f"  - Identical MultiIndex (dates & tickers): {same_index}")
print(f"  - Equal number of columns ({len(df_final_z.columns)}): {same_num_cols}")
print(f"  - Target '{TARGET}' present in both: {same_target}")

Structural Alignment Check:
  - Identical MultiIndex (dates & tickers): True
  - Equal number of columns (10): True
  - Target 'forward_return_21d' present in both: True


## 3. Data Preparation & Feature Extraction



### 3.1 Feature Selection (selección de predictores)


In [8]:
# Extract feature column names dynamically based on suffix
FEATURES_Z = [col for col in df_final_z.columns if col.endswith("_win_z")]
FEATURES_RANK = [col for col in df_final_rank.columns if col.endswith("_win_rank")]

print("Selected features for Z-Score model:")
print(FEATURES_Z)

print("\nSelected features for Percentile Rank model:")
print(FEATURES_RANK)

Selected features for Z-Score model:
['momentum_12_1_win_z', 'upside_volatility_win_z', 'log10_amihud_win_z']

Selected features for Percentile Rank model:
['momentum_12_1_win_rank', 'upside_volatility_win_rank', 'log10_amihud_win_rank']


### 3.2 Target Selection (forward_return_21d)

In [9]:
# Target variable for all models
TARGET = "forward_return_21d"

print(f"Target variable correctly set to: '{TARGET}'")

Target variable correctly set to: 'forward_return_21d'


### 3.3 Removal of Missing Observations (eliminación de NaNs en X e y)

In [10]:
# Drop missing values across features and target for each DataFrame
df_clean_z = df_final_z[FEATURES_Z + [TARGET]].dropna()
df_clean_rank = df_final_rank[FEATURES_RANK + [TARGET]].dropna()

# Align indices to guarantee 100% identical rows in both datasets
common_index = df_clean_z.index.intersection(df_clean_rank.index)

df_clean_z = df_clean_z.loc[common_index]
df_clean_rank = df_clean_rank.loc[common_index]

print("=== Complete Cases Summary ===")
print(f"Original rows: {len(df_final_z):,}")
print(f"Clean valid rows: {len(common_index):,}")
print(f"Dropped rows: {len(df_final_z) - len(common_index):,} ({((len(df_final_z) - len(common_index)) / len(df_final_z)) * 100:.2f}%)")

=== Complete Cases Summary ===
Original rows: 1,749,937
Clean valid rows: 1,627,370
Dropped rows: 122,567 (7.00%)


### 3.4 Final Modeling Datasets

In [11]:
# Final feature and target objects for Z-Score model
X_z = df_clean_z[FEATURES_Z]
y_z = df_clean_z[TARGET]

# Final feature and target objects for Percentile Rank model
X_rank = df_clean_rank[FEATURES_RANK]
y_rank = df_clean_rank[TARGET]

print("=== Final Datasets Ready for Modeling ===")
print(f"X_z shape: {X_z.shape} | y_z shape: {y_z.shape}")
print(f"X_rank shape: {X_rank.shape} | y_rank shape: {y_rank.shape}")
    
# Verification Check
dates = X_z.index.get_level_values("date")

print("\n=== Additional Verification ===")
print(f"First available date: {dates.min().strftime('%Y-%m-%d')}")
print(f"Last available date:  {dates.max().strftime('%Y-%m-%d')}")

=== Final Datasets Ready for Modeling ===
X_z shape: (1627370, 3) | y_z shape: (1627370,)
X_rank shape: (1627370, 3) | y_rank shape: (1627370,)

=== Additional Verification ===
First available date: 2011-01-03
Last available date:  2024-11-27


## 4. Walk-Forward Validation Strategy



### 4.1 Definición del esquema temporal de validación (Train / Validation / Test)

1. Segmentación del Dataset y Cronograma

    El dataset histórico (03/01/2011 al presente) se estructura en tres tramos operativos para aislar el desarrollo de la prueba final:

    - **Periodo de Desarrollo (Train / Validation)**: 03/01/2011 – 27/11/2024 ($\approx 14$ años).

    - **Búfer de Transición (Aislamiento)**: 28/11/2024 – mediados de enero de 2025.

    - **Backtest Out-of-Sample (OOS / Test)**: Desde mediados de enero de 2025 en adelante.

2. Optimización de Hiperparámetros: CPCV por Parejas

    Dentro del periodo de desarrollo se implementa un Combinatorial Purged Cross-Validation (CPCV) (López de Prado):

    - **Estructura de Bloques**: Muestra dividida en $N = 7$ bloques contiguos de $\approx 2$ años.

    - **Configuración por Parejas ($\varphi = 2$)**: En cada iteración, $2$ bloques se destinan a Validation y $5$ a Train, generando $\binom{7}{2} = 21$ combinaciones y 6 rutas sintéticas completas de backtest para evaluar la estabilidad del modelo (PBO, DSR).

3. Protocolo de Purga y Embargo (Gestión de Fronteras)

    Los recortes de información se aplican exclusivamente sobre el set de Entrenamiento (Train); los bloques de Validation permanecen intactos.

    - **Frontera Train $\rightarrow$ Validation (Purga)**: Se eliminan los últimos 21 días hábiles de Train, igual al horizonte de predicción de las etiquetas ($h = 21$), eliminando el Data Leakage.

    - **Frontera Validation $\rightarrow$ Train (Embargo)**: Se recortan los primeros 11 días hábiles de Train para neutralizar la autocorrelación residual y la memoria del mercado.

    - **Continuidad de Bloques (0 días)**:

        - Train $\rightarrow$ Train: Concatenación continua sin recortes (los modelos tabulares no secuenciales tratan las muestras como observaciones independientes).
    
        - Validation $\rightarrow$ Validation: Si son contiguos, se consolidan en un único bloque de evaluación sin purgas intermedias; si están separados, cada uno actúa como una isla con sus respectivas fronteras con Train.
        
4. Búfer de Separación y Backtest Operativo (Walk-Forward)

    - **Búfer Final**: Aplica 21 días naturales de Purga en diciembre de 2024 (vida útil de etiquetas) y 15 días naturales de Embargo ($\approx 11$ días hábiles) a inicios de enero de 2025 contra distorsiones de fin de año.
    
    - **Evaluación OOS (Test)**: Se ejecuta desde mediados de enero de 2025 mediante Walk-Forward con Ventana Expandida (Expanding Window), reentrenando progresivamente con todo el histórico disponible desde 2011 y aplicando el protocolo de Purga y Embargo en cada hito.


In [12]:
from src.models.utils import CombinatorialPurgedCV

# Instantiate the CPCV splitter
cpcv = CombinatorialPurgedCV(
    n_blocks=7,
    k_validation=2,
    purge_window=21,
    embargo_window=11,
)

# Generate all train/validation splits
splits = list(cpcv.split(X_z))

print("=== CPCV Sanity Check ===")
print(f"Number of CPCV folds: {len(splits)}")

train_idx, val_idx = splits[0]
print(f"Fold 01 -> Train observations: {len(train_idx):,} | Validation observations: {len(val_idx):,}")

=== CPCV Sanity Check ===
Number of CPCV folds: 21
Fold 01 -> Train observations: 1,185,716 | Validation observations: 436,671


El chequeo técnico confirma la correcta construcción del esquema CPCV en 21 combinaciones temporales ($\binom{7}{2}$). En el primer fold, el conjunto de entrenamiento cuenta con 1.185.716 observaciones (~73,1%) y el de validación con 436.671 (~26,9%). 

La pequeña desviación respecto a la proporción teórica pura (5/7 vs 2/7) refleja la aplicación efectiva de la purga de 21 días hábiles y el embargo de 11 días hábiles en las fronteras de entreno, un ajuste necesario que elimina por completo el solapamiento del forward return y el sesgo de anticipación (look-ahead bias) manteniendo la ventana de validación íntegra.

### 4.2 Control de solapamiento y prevención de Look-Ahead Bias (Purging window de 21 días)

Con el objetivo de verificar que la implementación del esquema **Combinatorial Purged Cross-Validation (CPCV)** respeta estrictamente las restricciones temporales definidas, se realiza una auditoría automática sobre la totalidad de los *folds* generados. En cada combinación de entrenamiento y validación se identifican los segmentos continuos de validación y se comprueba que todas las fronteras **Train → Validation** incorporan una **Purga (Purge)** de exactamente **21 días hábiles**, equivalente al horizonte de predicción de la variable objetivo, evitando así cualquier forma de *Look-Ahead Bias* o *Data Leakage*. De forma análoga, se verifica que todas las fronteras **Validation → Train** respetan un **Embargo** de **11 días hábiles**, reduciendo la posible dependencia temporal entre ambos conjuntos.

La auditoría se realiza utilizando exclusivamente el calendario de negociación del mercado (*trading days*), garantizando que tanto la identificación de los bloques contiguos de validación como el cálculo de las ventanas de Purga y Embargo son completamente independientes del calendario natural. Finalmente, se emplean comprobaciones automáticas mediante sentencias `assert`, de forma que cualquier incumplimiento del protocolo provoca la interrupción inmediata de la ejecución, certificando que los **21 folds** generados cumplen íntegramente el esquema temporal de validación definido para el proyecto.


In [13]:
# Extract the complete trading calendar
all_dates = (
    pd.Series(X_z.index.get_level_values("date").unique())
    .sort_values()
    .reset_index(drop=True)
)

# Create a lookup table: trading date -> position in calendar
date_to_pos = {date: pos for pos, date in enumerate(all_dates)}

# Audit every CPCV split
for fold, (train_idx, val_idx) in enumerate(splits, start=1):

    # Extract unique trading dates
    train_dates = (
        pd.Series(X_z.iloc[train_idx].index.get_level_values("date").unique())
        .sort_values()
        .reset_index(drop=True)
    )

    val_dates = (
        pd.Series(X_z.iloc[val_idx].index.get_level_values("date").unique())
        .sort_values()
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # Identify contiguous validation segments using trading-day positions
    # -------------------------------------------------------------------------

    val_positions = val_dates.map(date_to_pos)

    segments = []

    start = 0

    for i in range(1, len(val_positions)):

        # A new segment starts whenever trading days are no longer consecutive
        if val_positions.iloc[i] != val_positions.iloc[i - 1] + 1:

            segments.append((start, i - 1))
            start = i

    segments.append((start, len(val_positions) - 1))

    # -------------------------------------------------------------------------
    # Verify purge and embargo at every Train-Validation boundary
    # -------------------------------------------------------------------------

    for start_idx, end_idx in segments:

        segment_start = val_dates.iloc[start_idx]
        segment_end = val_dates.iloc[end_idx]

        # ===========================
        # Purge audit
        # ===========================

        train_before = train_dates[train_dates < segment_start]

        if not train_before.empty:

            last_train = train_before.max()

            purge_gap = (
                date_to_pos[segment_start]
                - date_to_pos[last_train]
                - 1
            )

            assert purge_gap == 21, (
                f"Fold {fold}: expected purge gap of 21 trading days, "
                f"found {purge_gap}."
            )

        # ===========================
        # Embargo audit
        # ===========================

        train_after = train_dates[train_dates > segment_end]

        if not train_after.empty:

            first_train = train_after.min()

            embargo_gap = (
                date_to_pos[first_train]
                - date_to_pos[segment_end]
                - 1
            )

            assert embargo_gap == 11, (
                f"Fold {fold}: expected embargo gap of 11 trading days, "
                f"found {embargo_gap}."
            )

print("✓ All 21 CPCV folds successfully passed purge and embargo integrity checks.")

✓ All 21 CPCV folds successfully passed purge and embargo integrity checks.


### 4.3 Análisis de cobertura temporal y recuento de muestras por bloque

Una vez validada la correcta implementación del protocolo de Purga y Embargo, se analiza la distribución temporal de la muestra utilizada durante la validación. 

En primer lugar, se reconstruyen los siete bloques temporales que conforman el esquema **Combinatorial Purged Cross-Validation (CPCV)**, resumiendo para cada uno su intervalo temporal, número de días de negociación, observaciones totales y cobertura media de activos por sesión. 

Posteriormente, se cuantifica la asignación efectiva de observaciones a los conjuntos de **Train** y **Validation**, así como la reducción del tamaño muestral derivada de la aplicación conjunta de las ventanas de Purga y Embargo. 

Finalmente, se incorporan comprobaciones de integridad para verificar que los bloques reconstruidos cubren la totalidad del periodo histórico sin solapamientos ni pérdidas de información.


In [14]:
# Extract the complete trading calendar
dates_series = (
    pd.Series(X_z.index.get_level_values("date").unique())
    .sort_values()
    .reset_index(drop=True)
)

n_dates = len(dates_series)
total_obs_dataset = len(X_z)

# Reconstruct temporal block boundaries
block_bounds = np.linspace(0, n_dates, cpcv.n_blocks + 1, dtype=int)

block_summary = []

for b in range(cpcv.n_blocks):

    start_idx, end_idx = block_bounds[b], block_bounds[b + 1]
    block_dates = dates_series.iloc[start_idx:end_idx]

    start_date = block_dates.min()
    end_date = block_dates.max()

    # Select all observations belonging to the current temporal block
    mask = X_z.index.get_level_values("date").isin(block_dates)

    n_obs = mask.sum()
    n_trading_days = len(block_dates)
    avg_assets_per_day = n_obs / n_trading_days

    block_summary.append({
        "Block": b + 1,
        "Start Date": start_date.strftime("%Y-%m-%d"),
        "End Date": end_date.strftime("%Y-%m-%d"),
        "Trading Days": n_trading_days,
        "Observations": n_obs,
        "Dataset (%)": 100 * n_obs / total_obs_dataset,
        "Avg Assets per Day": avg_assets_per_day,
    })

df_blocks = pd.DataFrame(block_summary)

# Display block-level summary
display(
    df_blocks
    .map(
        lambda x: (
            f"{x:.3g}"
            if isinstance(x, (int, float, np.integer, np.floating))
            else x
        )
    )
    .style.hide(axis="index")
)

# ------------------------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------------------------

# Verify that all trading days are covered exactly once
assert df_blocks["Trading Days"].sum() == len(dates_series)

# Verify that all observations belong to one and only one block
assert df_blocks["Observations"].sum() == total_obs_dataset

# ------------------------------------------------------------------------------
# CPCV sample allocation summary
# ------------------------------------------------------------------------------

avg_train_obs = np.mean([len(train_idx) for train_idx, _ in splits])
avg_val_obs = np.mean([len(val_idx) for _, val_idx in splits])

effective_sample_reduction = (
    1
    - (avg_train_obs + avg_val_obs) / total_obs_dataset
) * 100

df_cpcv_summary = pd.DataFrame(
    {
        "Total Observations": [total_obs_dataset],
        "Avg Train / Fold": [avg_train_obs],
        "Avg Validation / Fold": [avg_val_obs],
        "Effective Sample Reduction (%)": [effective_sample_reduction],
    }
)

# Display CPCV allocation summary
display(
    df_cpcv_summary
    .map(
        lambda x: (
            f"{x:.3g}"
            if isinstance(x, (int, float, np.integer, np.floating))
            else x
        )
    )
    .style.hide(axis="index")
)

Block,Start Date,End Date,Trading Days,Observations,Dataset (%),Avg Assets per Day
1,2011-01-03,2012-12-27,500,2.15e+05,13.2,429
2,2012-12-28,2014-12-22,500,2.22e+05,13.6,444
3,2014-12-23,2016-12-15,500,2.29e+05,14.1,458
4,2016-12-16,2018-12-12,500,2.34e+05,14.4,468
5,2018-12-13,2020-12-07,500,2.38e+05,14.6,476
6,2020-12-08,2022-12-01,500,2.43e+05,14.9,487
7,2022-12-02,2024-11-27,500,2.46e+05,15.1,493


Total Observations,Avg Train / Fold,Avg Validation / Fold,Effective Sample Reduction (%)
1.63e+06,1.14e+06,4.65e+05,1.31


El análisis de cobertura confirma una partición temporal perfectamente homogénea del historial de 3.500 días hábiles en 7 bloques idénticos de 500 días de negociación cada uno. A lo largo del periodo se observa un crecimiento lineal y constante en la cobertura diaria de activos, pasando de 429 empresas por día en el primer bloque a 493 en el séptimo, lo que refleja la expansión natural del universo computable y asigna de forma progresiva un peso ligeramente mayor a los datos más recientes. 

Por su parte, la aplicación estricta de las fronteras de purga y embargo conlleva una reducción efectiva de muestra de tan solo el 1,31% por combinación, lo que demuestra la eficiencia del algoritmo al fusionar bloques de validación contiguos. En definitiva, este mínimo descarte de información permite eliminar por completo el sesgo por solapamiento en el retorno a 21 días (*data leakage*) garantizando al mismo tiempo una masa crítica de entrenamiento y la continuidad total de las ventanas de evaluación.

## 5. Definición de Métricas de Evaluación

### 5.1 Métricas de Error (RMSE, MAE)

Para evaluar la precisión cuantitativa de los modelos en la predicción del retorno a 21 días ($y_{t+21}$), se emplean el Error Cuadrático Medio de la Raíz (RMSE) y el Error Absoluto Medio (MAE).

El RMSE actúa como la función de pérdida principal durante el ajuste de los modelos, ya que penaliza cuadráticamente las desviaciones de gran magnitud. En un entorno financiero, esta penalización asimétrica es crítica para evitar modelos con predicciones erráticas durante periodos de alta volatilidad o publicaciones de resultados. Por su parte, el MAE proporciona una medida lineal del error promedio en condiciones normales de mercado, ofreciendo una referencia robusta y menos sensible a valores atípicos (outliers).

La comparación conjunta entre ambas métricas permite auditar la presencia de colas anchas en los residuos del modelo: una discrepancia elevada entre el RMSE y el MAE señalará una mayor vulnerabilidad del algoritmo a desviaciones puntuales extremas en el panel de activos.

### 5.2 Métricas Financieras de Ranking (Information Coefficient - IC / Rank IC, Information Ratio)

En la modelización *cross-sectional* de paneles financieros, la prioridad del algoritmo es **ordenar correctamente los activos** de mejor a peor rendimiento esperado, más allá de la precisión cuantitativa del valor predicho. Para evaluar la calidad y consistencia de esta ordenación a lo largo del tiempo, empleamos el coeficiente de información diario mediante Spearman (Rank IC) como métrica primaria por su inmunidad a valores atípicos, complementado por Pearson (IC) para auditar posibles distorsiones extremas.

La serie temporal de IC diaria se sintetiza a través del **Information Ratio ($IR_{IC}$)**, que mide la estabilidad del alfa generado penalizando la volatilidad de la predicción, el **Hit Rate (%)**, que indica la frecuencia de días con ordenación acertada, y el **estadístico $t$**, que valida matemáticamente que la capacidad predictiva es estadísticamente significativa ($\vert{}t\vert{} > 2,0$) y no fruto del azar muestral.



### 5.3 Construcción del Evaluador Modular (Función de scoring)

Para operacionalizar la evaluación de manera eficiente y evitar duplicidad de código durante la optimización de modelos y la validación en los 21 folds del CPCV, construimos una función evaluadora modular (evaluate_predictions). Esta arquitectura unifica en un único punto de entrada el cálculo de las métricas de error cuantitativo (RMSE, MAE) y las de ordenación cross-sectional (IC, $IR_{IC}$, Hit Rate y $t$-stat). 

Al permitir la selección dinámica del método de correlación (Spearman o Pearson) y renombrar automáticamente las métricas resultantes, la función estandariza el vector de rendimiento de cualquier experimento, facilitando la comparación directa entre algoritmos y el registro centralizado de resultados.

## 6. Default Model Benchmark

Para evaluar la capacidad predictiva inherente a cada arquitectura sin sesgos introducidos por el ajuste de parámetros, esta sección establece el benchmark inicial entrenando todos los modelos bajo sus configuraciones por defecto. Utilizando la misma infraestructura de Combinatorial Purged Cross-Validation (CPCV), se analizan en igualdad de condiciones desde la Regresión Lineal pura hasta los algoritmos no lineales avanzados (Ridge, Random Forest, XGBoost y LightGBM). 

El objetivo es medir el potencial estructural base de cada familia de modelos y su comportamiento ante distintas transformaciones de los datos (Z-Score frente a Percentile Rank).

### 6.1 Generic Training Function

Para coordinar la ejecución de los modelos a través de la validación cruzada combinatoria, se ha implementado una pipeline genérica de entrenamiento (run_cpcv_training). Esta función abstrae el bucle de ajuste y predicción iterando sobre los 21 folds del esquema CPCV. En cada iteración, aísla los conjuntos de entrenamiento y validación, ajusta el algoritmo y genera las predicciones fuera de muestra (OOF). 

El proceso devuelve un registro ordenado cronológicamente con todas las predicciones de validación, el detalle del tamaño muestral y rendimiento por fold, y una tabla agregada que incluye la media y la desviación estándar de cada métrica. Esta agregación es vital, ya que permite evaluar tanto la precisión absoluta del modelo como su estabilidad y robustez frente a distintos regímenes de mercado.

### 6.2 Baseline Model: Linear Regression

In [17]:
from sklearn.linear_model import LinearRegression

# =============================================================================
# Baseline Model: Linear Regression
# =============================================================================

# Z-Score Dataset
oof_preds_lr_z, fold_metrics_lr_z, agg_metrics_lr_z = run_cpcv_training(
    model_cls=LinearRegression,
    model_params={},
    X=X_z,
    y=y_z,
    splits=splits,
    rank_method="spearman",
)

# Percentile Rank Dataset
oof_preds_lr_rank, fold_metrics_lr_rank, agg_metrics_lr_rank = run_cpcv_training(
    model_cls=LinearRegression,
    model_params={},
    X=X_rank,
    y=y_rank,
    splits=splits,
    rank_method="spearman",
)

# =============================================================================
# Results
# =============================================================================

print("BASELINE MODEL: LINEAR REGRESSION")
print("=" * 80)

# -----------------------------------------------------------------------------
# Z-Score Results
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("Z-SCORE NORMALIZATION")
print("-" * 80)

print(pd.Series(agg_metrics_lr_z).to_string())

# -----------------------------------------------------------------------------
# Percentile Rank Results
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("PERCENTILE RANK NORMALIZATION")
print("-" * 80)

print(pd.Series(agg_metrics_lr_rank).to_string())

print("\n" + "=" * 80)
print("END OF BASELINE EVALUATION")
print("=" * 80)


BASELINE MODEL: LINEAR REGRESSION

--------------------------------------------------------------------------------
Z-SCORE NORMALIZATION
--------------------------------------------------------------------------------
RMSE                          0.086159
MAE                           0.061101
Rank IC Mean                  0.033320
Rank IC Median                0.035376
Rank IC Std                   0.201028
Rank IC Information Ratio     0.175509
Rank IC Hit Rate (%)         56.385714
Rank IC t-statistic           5.550096

--------------------------------------------------------------------------------
PERCENTILE RANK NORMALIZATION
--------------------------------------------------------------------------------
RMSE                          0.086185
MAE                           0.061079
Rank IC Mean                  0.034321
Rank IC Median                0.037610
Rank IC Std                   0.196276
Rank IC Information Ratio     0.182704
Rank IC Hit Rate (%)         56.952381
Ran

### 6.3 Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

# =============================================================================
# Ridge Regression
# =============================================================================

# Z-Score Dataset
# -----------------------------------------------------------------------------

oof_preds_ridge_z, fold_metrics_ridge_z, agg_metrics_ridge_z = (
    run_cpcv_training(
        model_cls=Ridge,
        model_params={},
        X=X_z,
        y=y_z,
        splits=splits,
        rank_method="spearman",
    )
)


# Percentile Rank Dataset
# -----------------------------------------------------------------------------

oof_preds_ridge_rank, fold_metrics_ridge_rank, agg_metrics_ridge_rank = (
    run_cpcv_training(
        model_cls=Ridge,
        model_params={},
        X=X_rank,
        y=y_rank,
        splits=splits,
        rank_method="spearman",
    )
)


# =============================================================================
# Results
# =============================================================================

print("\n")
print("=" * 80)
print("RIDGE REGRESSION")
print("=" * 80)


# -----------------------------------------------------------------------------
# Z-Score Results
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("Z-SCORE NORMALIZATION")
print("-" * 80)

print(agg_metrics_ridge_z.to_string())


# -----------------------------------------------------------------------------
# Percentile Rank Results
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("PERCENTILE RANK NORMALIZATION")
print("-" * 80)

print(agg_metrics_ridge_rank.to_string())


# =============================================================================
# End of Evaluation
# =============================================================================

print("\n" + "=" * 80)
print("END OF RIDGE REGRESSION EVALUATION")
print("=" * 80)



RIDGE REGRESSION

--------------------------------------------------------------------------------
Z-SCORE NORMALIZATION
--------------------------------------------------------------------------------
          RMSE       MAE  Rank IC Mean  Rank IC Median  Rank IC Std  Rank IC Information Ratio  Rank IC Hit Rate (%)  Rank IC t-statistic
mean  0.086159  0.061101      0.033320        0.035376     0.201028                   0.175509             56.385714             5.550096
std   0.010806  0.006362      0.019588        0.025796     0.023941                   0.110576              4.501254             3.496728

--------------------------------------------------------------------------------
PERCENTILE RANK NORMALIZATION
--------------------------------------------------------------------------------
          RMSE       MAE  Rank IC Mean  Rank IC Median  Rank IC Std  Rank IC Information Ratio  Rank IC Hit Rate (%)  Rank IC t-statistic
mean  0.086185  0.061079      0.034321         0.03

### 6.4 Random Forest

Random Forest se evaluó inicialmente como candidato dentro del benchmark de modelos. Sin embargo, su coste computacional resultó desproporcionadamente elevado para el tamaño de nuestro dataset: un único fold de CPCV requiere aproximadamente 38 segundos incluso utilizando únicamente 10 árboles.

Dado que el proceso completo implica 21 folds y dos conjuntos de normalización (Z-Score y Percentile Rank), ejecutar el benchmark completo supondría un coste considerablemente superior al de otros modelos basados en árboles.

Por este motivo, Random Forest se excluye provisionalmente del benchmark principal, no por considerar que su capacidad predictiva sea inferior, sino por su elevado coste computacional en relación con los demás candidatos. XGBoost y LightGBM se evaluarán a continuación bajo el mismo esquema de validación.

Random Forest podrá recuperarse posteriormente como extensión del análisis si los resultados de los modelos seleccionados justifican una comparación adicional.


### 6.5 XGBoost

In [36]:
from xgboost import XGBRegressor

# =============================================================================
# XGBoost Regression - Z-Score
# =============================================================================

oof_preds_xgb_z, fold_metrics_xgb_z, agg_metrics_xgb_z = (
    run_cpcv_training(
        model_cls=XGBRegressor,
        model_params={
            "random_state": SEED,
        },
        X=X_z,
        y=y_z,
        splits=splits,
        rank_method="spearman",
    )
)


# =============================================================================
# Results
# =============================================================================

print("\n")
print("=" * 80)
print("XGBOOST REGRESSION - Z-SCORE")
print("=" * 80)

print("\n" + "-" * 80)
print("Z-SCORE NORMALIZATION")
print("-" * 80)

print_metrics(agg_metrics_xgb_z)

print("\n" + "=" * 80)
print("END OF XGBOOST Z-SCORE EVALUATION")
print("=" * 80)



XGBOOST REGRESSION - Z-SCORE

--------------------------------------------------------------------------------
Z-SCORE NORMALIZATION
--------------------------------------------------------------------------------
RMSE                             0.0882
MAE                              0.0622
Rank IC Mean                     0.0110
Rank IC Median                   0.0127
Rank IC Std                      0.0869
Rank IC Information Ratio        0.1318
Rank IC Hit Rate (%)              55.48%
Rank IC t-statistic              4.1687

END OF XGBOOST Z-SCORE EVALUATION


In [37]:
from xgboost import XGBRegressor

# =============================================================================
# XGBoost Regression - Percentile Rank
# =============================================================================

oof_preds_xgb_rank, fold_metrics_xgb_rank, agg_metrics_xgb_rank = (
    run_cpcv_training(
        model_cls=XGBRegressor,
        model_params={
            "random_state": SEED,
        },
        X=X_rank,
        y=y_rank,
        splits=splits,
        rank_method="spearman",
    )
)


# =============================================================================
# Results
# =============================================================================

print("\n")
print("=" * 80)
print("XGBOOST REGRESSION - PERCENTILE RANK")
print("=" * 80)

print("\n" + "-" * 80)
print("PERCENTILE RANK NORMALIZATION")
print("-" * 80)

print_metrics(agg_metrics_xgb_rank)

print("\n" + "=" * 80)
print("END OF XGBOOST PERCENTILE RANK EVALUATION")
print("=" * 80)



XGBOOST REGRESSION - PERCENTILE RANK

--------------------------------------------------------------------------------
PERCENTILE RANK NORMALIZATION
--------------------------------------------------------------------------------
RMSE                             0.0881
MAE                              0.0620
Rank IC Mean                     0.0122
Rank IC Median                   0.0125
Rank IC Std                      0.0939
Rank IC Information Ratio        0.1344
Rank IC Hit Rate (%)              54.94%
Rank IC t-statistic              4.2516

END OF XGBOOST PERCENTILE RANK EVALUATION


### 6.6 LightGBM

In [38]:
from lightgbm import LGBMRegressor

# =============================================================================
# LightGBM Regression - Z-Score
# =============================================================================

oof_preds_lgbm_z, fold_metrics_lgbm_z, agg_metrics_lgbm_z = (
    run_cpcv_training(
        model_cls=LGBMRegressor,
        model_params={
            "random_state": SEED,
        },
        X=X_z,
        y=y_z,
        splits=splits,
        rank_method="spearman",
    )
)


# =============================================================================
# Results
# =============================================================================

print("\n")
print("=" * 80)
print("LIGHTGBM REGRESSION - Z-SCORE")
print("=" * 80)

print("\n" + "-" * 80)
print("Z-SCORE NORMALIZATION")
print("-" * 80)

print_metrics(agg_metrics_lgbm_z)

print("\n" + "=" * 80)
print("END OF LIGHTGBM Z-SCORE EVALUATION")
print("=" * 80)



LIGHTGBM REGRESSION - Z-SCORE

--------------------------------------------------------------------------------
Z-SCORE NORMALIZATION
--------------------------------------------------------------------------------
RMSE                             0.0869
MAE                              0.0615
Rank IC Mean                     0.0249
Rank IC Median                   0.0306
Rank IC Std                      0.1351
Rank IC Information Ratio        0.1930
Rank IC Hit Rate (%)              58.40%
Rank IC t-statistic              6.1046

END OF LIGHTGBM Z-SCORE EVALUATION


In [39]:
from lightgbm import LGBMRegressor

# =============================================================================
# LightGBM Regression - Percentile Rank
# =============================================================================

oof_preds_lgbm_rank, fold_metrics_lgbm_rank, agg_metrics_lgbm_rank = (
    run_cpcv_training(
        model_cls=LGBMRegressor,
        model_params={
            "random_state": SEED,
        },
        X=X_rank,
        y=y_rank,
        splits=splits,
        rank_method="spearman",
    )
)


# =============================================================================
# Results
# =============================================================================

print("\n")
print("=" * 80)
print("LIGHTGBM REGRESSION - PERCENTILE RANK")
print("=" * 80)

print("\n" + "-" * 80)
print("PERCENTILE RANK NORMALIZATION")
print("-" * 80)

print_metrics(agg_metrics_lgbm_rank)

print("\n" + "=" * 80)
print("END OF LIGHTGBM PERCENTILE RANK EVALUATION")
print("=" * 80)



LIGHTGBM REGRESSION - PERCENTILE RANK

--------------------------------------------------------------------------------
PERCENTILE RANK NORMALIZATION
--------------------------------------------------------------------------------
RMSE                             0.0869
MAE                              0.0614
Rank IC Mean                     0.0259
Rank IC Median                   0.0318
Rank IC Std                      0.1461
Rank IC Information Ratio        0.1864
Rank IC Hit Rate (%)              57.90%
Rank IC t-statistic              5.8955

END OF LIGHTGBM PERCENTILE RANK EVALUATION


## 7. Default Model Comparison

En esta sección se consolida y contrasta de forma transversal el rendimiento del benchmark. A través del análisis comparativo entre transformaciones, el ranking global por Rank IC e $IR_{IC}$, y la evaluación de la estabilidad temporal entre folds, se identifican los patrones de comportamiento de cada algoritmo. Este diagnóstico cuantitativo servirá de filtro fundamentado para seleccionar únicamente los modelos candidatos que pasarán a la fase de optimización computacional.

Una vez ejecutados todos los modelos bajo la configuración por defecto y evaluadas ambas representaciones de los datos, consolidamos sus resultados en una única tabla. Esta tabla recoge todas las métricas obtenidas durante la validación CPCV y se almacena en formato Parquet para poder reutilizarla posteriormente sin necesidad de volver a ejecutar los modelos.

In [52]:
from src.models.utils import extract_mean_metrics

benchmark_results = [

    {
        "Model": "Linear Regression",
        "Scaling / Representation": "Z-Score",
        **extract_mean_metrics(agg_metrics_lr_z),
    },

    {
        "Model": "Linear Regression",
        "Scaling / Representation": "Percentile Rank",
        **extract_mean_metrics(agg_metrics_lr_rank),
    },

    {
        "Model": "Ridge",
        "Scaling / Representation": "Z-Score",
        **extract_mean_metrics(agg_metrics_ridge_z),
    },

    {
        "Model": "Ridge",
        "Scaling / Representation": "Percentile Rank",
        **extract_mean_metrics(agg_metrics_ridge_rank),
    },

    {
        "Model": "XGBoost",
        "Scaling / Representation": "Z-Score",
        **extract_mean_metrics(agg_metrics_xgb_z),
    },

    {
        "Model": "XGBoost",
        "Scaling / Representation": "Percentile Rank",
        **extract_mean_metrics(agg_metrics_xgb_rank),
    },

    {
        "Model": "LightGBM",
        "Scaling / Representation": "Z-Score",
        **extract_mean_metrics(agg_metrics_lgbm_z),
    },

    {
        "Model": "LightGBM",
        "Scaling / Representation": "Percentile Rank",
        **extract_mean_metrics(agg_metrics_lgbm_rank),
    },
]

df_benchmark = pd.DataFrame(benchmark_results)
df_benchmark.to_parquet("../data/model_results/df_benchmark.parquet")

In [18]:
df_benchmark = pd.read_parquet("../data/model_results/df_benchmark.parquet")

### 7.1 Análisis Integral del Default Model Benchmark

#### 7.1.1 Capacidad Predictiva

La capacidad predictiva se evaluará principalmente mediante el Mean Rank IC, ya que nuestro objetivo es medir hasta qué punto el modelo consigue ordenar correctamente los activos según su rentabilidad futura. Se complementará con el Rank IC IR, que incorpora la variabilidad del IC y permite valorar la consistencia temporal de dicha capacidad predictiva, y con el Hit Rate, que indica el porcentaje de periodos en los que el Rank IC es positivo.

In [57]:
# =============================================================================
# 7.1.1 Predictive Capacity
# =============================================================================

df_predictive_capacity = df_benchmark[
    [
        "Model",
        "Scaling / Representation",
        "Rank IC Mean",
        "Rank IC Information Ratio",
        "Rank IC Hit Rate (%)",
    ]
].copy()

print("\n" + "=" * 100)
print("7.1.1 PREDICTIVE CAPACITY")
print("=" * 100)

display(
    df_predictive_capacity.style
    .hide(axis="index")
)


7.1.1 PREDICTIVE CAPACITY


Model,Scaling / Representation,Rank IC Mean,Rank IC Information Ratio,Rank IC Hit Rate (%)
Linear Regression,Z-Score,0.033320,0.175509,56.385714
Linear Regression,Percentile Rank,0.034321,0.182704,56.952381
Ridge,Z-Score,0.033320,0.175509,56.385714
Ridge,Percentile Rank,0.034321,0.182704,56.952381
XGBoost,Z-Score,0.011029,0.131827,55.480952
XGBoost,Percentile Rank,0.012215,0.134447,54.942857
LightGBM,Z-Score,0.024878,0.193046,58.400000
LightGBM,Percentile Rank,0.025933,0.186433,57.904762


La evaluación de la capacidad predictiva en la configuración por defecto revela un contraste metodológico clave entre la familia lineal y los algoritmos basados en árboles de decisión.

La Regresión Lineal (tanto MCO como Ridge) lidera en fuerza de ordenación cross-sectional, alcanzando el Rank IC Mean más alto del panel con un 0.0343 bajo la representación Percentile Rank. Este resultado confirma que la señal subyacente de los factores presenta una estructura predominantemente lineal en su estado base. Por su parte, LightGBM destaca en consistencia y estabilidad temporal: aunque su IC medio es más moderado (0.0259), logra el mayor Hit Rate direccional de la comparativa con un 58.40% y el mejor Rank IC IR (0.1930) sobre datos Z-Score, posicionándose como el algoritmo más fiable a la hora de mantener un IC positivo a través de las distintas particiones temporales.

En el extremo opuesto, XGBoost registra el desempeño más débil con un Rank IC Mean de apenas 0.0110 - 0.0122, afectado por la tendencia de sus parámetros por defecto (como max_depth=6) a memorizar el ruido inherente a las series financieras; este margen de mejora lo convierte en un candidato crítico para evaluar la ganancia por optimización en la Sección 8. Finalmente, la transformación a Percentile Rank demuestra un beneficio transversal en todas las arquitecturas, elevando sistemáticamente el Rank IC Mean respecto al Z-Score al mitigar el efecto de los valores extremos y uniformizar la distribución del panel de entrada.

#### 7.1.2 Significancia Estadística

Para evaluar la solidez estadística de los resultados utilizaremos principalmente el Rank IC t-statistic, que permite determinar si el IC medio observado presenta suficiente evidencia estadística frente a un valor nulo. Esta métrica se utilizará como complemento de las métricas de capacidad predictiva, y no como criterio único de selección.

In [58]:
# =============================================================================
# 7.1.2 Statistical Significance
# =============================================================================

df_statistical_significance = df_benchmark[
    [
        "Model",
        "Scaling / Representation",
        "Rank IC Mean",
        "Rank IC t-statistic",
    ]
].copy()

print("\n" + "=" * 100)
print("7.1.2 STATISTICAL SIGNIFICANCE")
print("=" * 100)

display(
    df_statistical_significance.style
    .hide(axis="index")
)


7.1.2 STATISTICAL SIGNIFICANCE


Model,Scaling / Representation,Rank IC Mean,Rank IC t-statistic
Linear Regression,Z-Score,0.033320,5.550096
Linear Regression,Percentile Rank,0.034321,5.777593
Ridge,Z-Score,0.033320,5.550096
Ridge,Percentile Rank,0.034321,5.777599
XGBoost,Z-Score,0.011029,4.168736
XGBoost,Percentile Rank,0.012215,4.251585
LightGBM,Z-Score,0.024878,6.104644
LightGBM,Percentile Rank,0.025933,5.895534


Todas las configuraciones superan con holgura no solo el umbral estadístico clásico ($\vert{}t\vert{} > 1.96$), sino también el criterio estricto exigido en la literatura financiera cuantitativa ($\vert{}t\vert{} > 3.00$), garantizando que la capacidad predictiva observada no es fruto del azar ni de un artefacto de la muestra. 

LightGBM bajo la representación Z-Score registra la mayor significancia estadística del panel con un $t$-stat insuperable de 6.1046, seguido muy de cerca por su versión Percentile Rank (5.8955). Esta cifra evidencia que, aunque su Rank IC absoluto es ligeramente menor que el de la regresión lineal, la señal de LightGBM destaca por una variabilidad sumamente reducida y una precisión excepcional a través de los folds. 

Por su parte, la Regresión Lineal y Ridge consolidan una significancia sobresaliente con valores de $t$-stat entre 5.5501 y 5.7776, respaldando la firmeza económica de su fuerza de ordenación. 

Finalmente, aunque XGBoost obtiene el valor más bajo de la comparativa, su $t$-stat de 4.1687 – 4.2516 demuestra que, incluso bajo una configuración por defecto propensa al sobreajuste, conserva una señal estadísticamente real y aprovechable.

#### 7.1.3 Robustez y Estabilidad

La robustez de los modelos se analizará mediante el Median Rank IC y el Rank IC Std. La mediana permitirá comprobar si el rendimiento medio está siendo representativo o está condicionado por determinados periodos extremos, mientras que la desviación estándar mostrará la variabilidad del IC entre los distintos periodos evaluados.

In [ ]:
# =============================================================================
# 7.1.3 Robustness and Stability
# =============================================================================

df_robustness = df_benchmark[
    [
        "Model",
        "Scaling / Representation",
        "Rank IC Mean",
        "Rank IC Median",
        "Rank IC Std",
        "Rank IC Information Ratio",
    ]
].copy()

print("\n" + "=" * 100)
print("7.1.3 ROBUSTNESS AND STABILITY")
print("=" * 100)

display(
    df_robustness.style
    .hide(axis="index")
)


7.1.3 ROBUSTNESS AND STABILITY


Model,Scaling / Representation,Rank IC Mean,Rank IC Median,Rank IC Std,Rank IC Information Ratio
Linear Regression,Z-Score,0.033320,0.035376,0.201028,0.175509
Linear Regression,Percentile Rank,0.034321,0.037610,0.196276,0.182704
Ridge,Z-Score,0.033320,0.035376,0.201028,0.175509
Ridge,Percentile Rank,0.034321,0.037610,0.196276,0.182704
XGBoost,Z-Score,0.011029,0.012668,0.086896,0.131827
XGBoost,Percentile Rank,0.012215,0.012477,0.093897,0.134447
LightGBM,Z-Score,0.024878,0.030633,0.135056,0.193046
LightGBM,Percentile Rank,0.025933,0.031849,0.146050,0.186433


En todas las arquitecturas evaluadas, la mediana del Rank IC supera sistemáticamente a la media, evidenciando una asimetría negativa moderada. Esto confirma que los resultados promedio no están inflados por periodos atípicos estallados al alza; al contrario, en un entorno de mercado habitual el rendimiento esperado tiende a ser ligeramente superior a la media declarada, viéndose esta reducida por episodios puntuales de fuerte cambio de régimen.

En términos de dispersión, los modelos de árboles presentan un perfil de riesgo temporal sensiblemente más acotado que la familia lineal. Mientras la Regresión Lineal y Ridge registran la mayor desviación estándar del panel (Rank IC Std $\approx 0.1963 - 0.2010$), LightGBM logra comprimir la volatilidad del IC hasta un 0.1351 – 0.1461. Esta notable estabilidad permite a LightGBM maximizar la ratio de información (Rank IC IR de 0.1930 en Z-Score), consolidándose como la arquitectura más consistente del benchmark. Por su parte, XGBoost exhibe la desviación estándar más baja de la comparativa (0.0869).

#### 7.1.4 Precisión de las Predicciones

La precisión numérica de las predicciones se evaluará mediante RMSE y MAE. Estas métricas permiten medir la magnitud de los errores cometidos por el modelo. Sin embargo, tendrán un carácter secundario, ya que el objetivo principal del modelo no es predecir exactamente el retorno futuro, sino conseguir un buen ordenamiento cross-sectional de los activos.

In [60]:
# =============================================================================
# 7.1.4 Prediction Accuracy
# =============================================================================

df_prediction_accuracy = df_benchmark[
    [
        "Model",
        "Scaling / Representation",
        "RMSE",
        "MAE",
    ]
].copy()

print("\n" + "=" * 100)
print("7.1.4 PREDICTION ACCURACY")
print("=" * 100)

display(
    df_prediction_accuracy.style
    .hide(axis="index")
)


7.1.4 PREDICTION ACCURACY


Model,Scaling / Representation,RMSE,MAE
Linear Regression,Z-Score,0.086159,0.061101
Linear Regression,Percentile Rank,0.086185,0.061079
Ridge,Z-Score,0.086159,0.061101
Ridge,Percentile Rank,0.086185,0.061079
XGBoost,Z-Score,0.088154,0.062164
XGBoost,Percentile Rank,0.088114,0.062049
LightGBM,Z-Score,0.086938,0.061464
LightGBM,Percentile Rank,0.086898,0.061385


Los resultados muestran una estrecha convergencia en el nivel de error cuadrático y absoluto entre todas las arquitecturas, situándose el RMSE global en el rango del 8.61% – 8.81% y el MAE alrededor del 6.10% – 6.21%, magnitudes acordes a la volatilidad implícita del mercado en horizontes mensuales. 

La Regresión Lineal y Ridge obtienen los errores más ajustados del panel (RMSE de 0.086159), beneficiándose de la optimización directa por mínimos cuadrados para minimizar la varianza de los residuos fuera de muestra. 

Por su parte, LightGBM se posiciona muy cerca del suelo lineal (RMSE de 0.086898), confirmando que su superioridad en Hit Rate e IR no se logra a costa de descalibrar las predicciones numéricas. En contraste, XGBoost registra los mayores márgenes de error (RMSE de 0.088154), coherente con la falta de regularización de sus parámetros por defecto. 

Finalmente, la elección entre Z-Score y Percentile Rank no genera variaciones estadísticamente significativas en el nivel de error, demostrando que la ganancia observada en Rank IC bajo la transformación porcentual es limpia y no distorsiona la escala del modelo.

#### 7.1.5 Comparación Z-Score vs Percentile Rank

Finalmente, se comparará el rendimiento de ambas representaciones utilizando conjuntamente Mean Rank IC, Rank IC IR, Hit Rate y t-statistic, complementadas por las métricas de robustez y error. El objetivo es determinar si alguna de las dos normalizaciones proporciona una ventaja consistente independientemente del modelo utilizado.

In [62]:
# =============================================================================
# 7.1.5 Z-Score vs Percentile Rank
# =============================================================================

# Separate representations
df_z = (
    df_benchmark[
        df_benchmark["Scaling / Representation"] == "Z-Score"
    ]
    .set_index("Model")
)

df_rank = (
    df_benchmark[
        df_benchmark["Scaling / Representation"] == "Percentile Rank"
    ]
    .set_index("Model")
)


# Calculate differences:
# Δ = Z-Score - Percentile Rank

df_representation_comparison = pd.DataFrame(
    {
        "Δ Rank IC Mean\n(Z - Rank)": (
            df_z["Rank IC Mean"] - df_rank["Rank IC Mean"]
        ),

        "Δ Rank IC IR\n(Z - Rank)": (
            df_z["Rank IC Information Ratio"]
            - df_rank["Rank IC Information Ratio"]
        ),

        "Δ Hit Rate (%)\n(Z - Rank)": (
            df_z["Rank IC Hit Rate (%)"]
            - df_rank["Rank IC Hit Rate (%)"]
        ),

        "Δ t-statistic\n(Z - Rank)": (
            df_z["Rank IC t-statistic"]
            - df_rank["Rank IC t-statistic"]
        ),
    }
).reset_index()


# Round numerical values
numeric_columns = df_representation_comparison.select_dtypes(
    include="number"
).columns

df_representation_comparison[numeric_columns] = (
    df_representation_comparison[numeric_columns].round(4)
)


# =============================================================================
# Display Results
# =============================================================================

print("\n" + "=" * 100)
print("7.1.5 Z-SCORE VS PERCENTILE RANK")
print("=" * 100)

print("\nΔ = Z-Score - Percentile Rank")
print("Positive values favor Z-Score; negative values favor Percentile Rank.\n")

display(
    df_representation_comparison.style
    .hide(axis="index")
)


7.1.5 Z-SCORE VS PERCENTILE RANK

Δ = Z-Score - Percentile Rank
Positive values favor Z-Score; negative values favor Percentile Rank.



Model,Δ Rank IC Mean (Z - Rank),Δ Rank IC IR (Z - Rank),Δ Hit Rate (%) (Z - Rank),Δ t-statistic (Z - Rank)
Linear Regression,-0.001000,-0.007200,-0.566700,-0.227500
Ridge,-0.001000,-0.007200,-0.566700,-0.227500
XGBoost,-0.001200,-0.002600,0.538100,-0.082800
LightGBM,-0.001100,0.006600,0.495200,0.209100


La transformación a Percentile Rank mejora de forma transversal y sistemática la fuerza de ordenación en todas las arquitecturas, generando un incremento en el Rank IC Mean de entre $+0.0010$ y $+0.0012$** ($\Delta$` de $-0.0010$ a $-0.0012$). Al comprimir los factores a un rango uniforme $[0, 1]$, esta representación atenúa el impacto de los valores extremos (outliers) y permite que tanto los modelos lineales como los árboles se enfoquen exclusivamente en la estructura ordinal de la muestra.

Sin embargo, el impacto en la estabilidad y la precisión direccional muestra una clara divergencia por familia de modelos:

- **Modelos Lineales (MCO y Ridge)**: La representación Percentile Rank es indiscutiblemente superior en todas las métricas. Además de elevar la media del IC, incrementa la ratio de información ($\Delta$ IR = -0.0072), la tasa de aciertos ($\Delta$ Hit Rate = -0.5667%) y la significancia estadística ($\Delta$ t-stat = -0.2275), demostrando que la uniformización de los factores ayuda a mitigar la varianza en modelos paramétricos.

- **Modelos de Gradient Boosting (LightGBM y XGBoost)**: Aunque el Percentile Rank mejora marginalmente la magnitud de su Rank IC, la transformación Z-Score preserva de forma mucho más eficiente la consistencia y significancia del modelo. LightGBM bajo Z-Score obtiene un mayor Hit Rate ($\Delta$ Hit Rate = +0.4952%), un IR superior ($\Delta$ IR = +0.0066) y su punto máximo de significancia estadística ($\Delta$ t-stat = +0.2091), alcanzando un $t$-stat absoluto de 6.1046. De igual modo, XGBoost logra un mejor acierto direccional bajo Z-Score ($\Delta$ Hit Rate = +0.5381%), confirmando que mantener las magnitudes y colas originales de la distribución favorece a los árboles en la detección de regímenes de mercado explícitos.

### 7.2 Ranking de modelos

#### 7.2.1 Construcción del Score Multicriterio

Para objetivar el ranking de los ocho experimentos, se evalúa el rendimiento mediante una puntuación compuesta que combina las tres dimensiones principales del modelo: la fuerza bruta de ordenación de los activos (Rank IC Mean, $50\%$), la consistencia temporal de la señal (Rank IC IR, $30\%$) y la frecuencia de períodos con un IC positivo (Hit Rate, $20\%$). Para homogeneizar las distintas escalas, cada métrica se transforma previamente a su rango percentil dentro del panel antes de calcular la suma ponderada:

$$Score = 0.50 \cdot Rank(IC_{Mean}) + 0.30 \cdot Rank(IC_{IR}) + 0.20 \cdot Rank(HitRate)$$

El $t$-statistic se excluye deliberadamente del cómputo para evitar redundancias con la ratio de información y se reserva como indicador complementario de significancia estadística.

In [63]:
# Copy benchmark results
df_score = df_benchmark.copy()


# -----------------------------------------------------------------------------
# 1. Convert performance metrics into percentile ranks
# -----------------------------------------------------------------------------

df_score["Rank IC Mean Score"] = (
    df_score["Rank IC Mean"]
    .rank(method="average", pct=True)
)

df_score["Rank IC IR Score"] = (
    df_score["Rank IC Information Ratio"]
    .rank(method="average", pct=True)
)

df_score["Hit Rate Score"] = (
    df_score["Rank IC Hit Rate (%)"]
    .rank(method="average", pct=True)
)


# -----------------------------------------------------------------------------
# 2. Calculate weighted multicriteria score
# -----------------------------------------------------------------------------

df_score["Multicriteria Score"] = (
    0.50 * df_score["Rank IC Mean Score"]
    + 0.30 * df_score["Rank IC IR Score"]
    + 0.20 * df_score["Hit Rate Score"]
)


# -----------------------------------------------------------------------------
# 3. Create final ranking
# -----------------------------------------------------------------------------

df_score = df_score.sort_values(
    "Multicriteria Score",
    ascending=False
).reset_index(drop=True)


df_score["Position"] = (
    df_score["Multicriteria Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -----------------------------------------------------------------------------
# 4. Select columns for the ranking
# -----------------------------------------------------------------------------

df_model_ranking = df_score[
    [
        "Position",
        "Model",
        "Scaling / Representation",
        "Rank IC Mean",
        "Rank IC Information Ratio",
        "Rank IC Hit Rate (%)",
        "Rank IC t-statistic",
        "Multicriteria Score",
    ]
].copy()


# -----------------------------------------------------------------------------
# 5. Round numerical values
# -----------------------------------------------------------------------------

numeric_columns = df_model_ranking.select_dtypes(
    include="number"
).columns

df_model_ranking[numeric_columns] = (
    df_model_ranking[numeric_columns].round(4)
)


# =============================================================================
# Display Results
# =============================================================================

print("\n" + "=" * 110)
print("7.2.1 MULTICRITERIA MODEL RANKING")
print("=" * 110)

print("\nScore composition:")
print("  Rank IC Mean           → 50%")
print("  Rank IC Information Ratio → 30%")
print("  Rank IC Hit Rate       → 20%")

print("\nPercentile ranking:")
print("  Higher values = better performance")

print("\n" + "-" * 110)

display(
    df_model_ranking.style
    .hide(axis="index")
)


7.2.1 MULTICRITERIA MODEL RANKING

Score composition:
  Rank IC Mean           → 50%
  Rank IC Information Ratio → 30%
  Rank IC Hit Rate       → 20%

Percentile ranking:
  Higher values = better performance

--------------------------------------------------------------------------------------------------------------


Position,Model,Scaling / Representation,Rank IC Mean,Rank IC Information Ratio,Rank IC Hit Rate (%),Rank IC t-statistic,Multicriteria Score
1,Ridge,Percentile Rank,0.034300,0.182700,56.952400,5.777600,0.862500
2,Linear Regression,Percentile Rank,0.034300,0.182700,56.952400,5.777600,0.762500
3,LightGBM,Z-Score,0.024900,0.193000,58.400000,6.104600,0.687500
3,LightGBM,Percentile Rank,0.025900,0.186400,57.904800,5.895500,0.687500
5,Ridge,Z-Score,0.033300,0.175500,56.385700,5.550100,0.612500
6,Linear Regression,Z-Score,0.033300,0.175500,56.385700,5.550100,0.512500
7,XGBoost,Percentile Rank,0.012200,0.134400,54.942900,4.251600,0.225000
8,XGBoost,Z-Score,0.011000,0.131800,55.481000,4.168700,0.150000


#### 7.2.2 Ranking Final

La selección de los modelos que avanzarán a la fase de ajuste de hiperparámetros se determina mediante tres criterios fundamentales: 

1. Pasarán los experimentos que lideren el ranking global según el Score Multicriterio. 

2. Se rescatará cualquier arquitectura no lineal que, aunque no obtenga la puntuación más alta, destaque de forma sobresaliente en dimensiones críticas como la consistencia temporal, el acierto direccional o la significancia estadística. 

3. Se garantiza la permanencia de al menos un modelo lineal y uno no lineal para evaluar en la siguiente fase si la optimización altera las conclusiones y la jerarquía observadas en el benchmark por defecto.

La evaluación del panel de modelos revela una clara jerarquía operativa encabezada por la arquitectura lineal. Ridge y la Regresión Lineal dominan la clasificación absoluta gracias a la transformación por Percentile Rank, la cual maximiza su capacidad de ordenación cross-sectional y demuestra que la señal lineal fundamental es excepcionalmente robusta una vez que se acotan los efectos de los valores extremos.

Por su parte, LightGBM se consolida como la alternativa más consistente y equilibrada del estudio. Independientemente de la normalización aplicada, el algoritmo se mantiene en la parte alta de la tabla: mientras que la variante por Percentile Rank aporta un mayor rendimiento medio en la señal, el tratamiento por Z-Score destaca por ofrecer el techo del benchmark en cuanto a estabilidad y significancia estadística.

En el extremo opuesto, XGBoost queda relegado a las últimas posiciones de la comparativa. En su configuración inicial, el modelo muestra una capacidad predictiva sensiblemente inferior a la del resto de sus competidores, sin llegar a capturar la estructura del mercado con la misma eficacia.

## 7.3 Selección de modelos candidatos para optimización

1. **Pasan Ridge y Regresión Lineal (Percentile Rank)**: Cumplen el Criterio 1 al situarse como los ganadores del Score multicriterio. Dado que sus métricas son prácticamente idénticas en el baseline, mantener ambos permitirá comprobar mediante Optuna si la regularización (L_2) de Ridge aporta valor adicional frente a la regresión lineal sin regularización.

2. **Pasa LightGBM (Z-Score y Percentile Rank)**: Cumplen el Criterio 2. Aunque quedan por detrás de los modelos lineales en el Score debido principalmente a un menor IC medio, presentan señales destacables de consistencia y significancia. La versión Z-Score obtiene el máximo Hit Rate, Rank IC IR y (t)-statistic del panel, mientras que Percentile Rank presenta el mayor Rank IC Mean dentro de la familia no lineal. Mantener ambas representaciones permitirá comprobar cómo responde cada una al proceso de optimización.


3. **Pasa XGBoost (Percentile Rank)**: Cumple el Criterio 3. Se selecciona la versión Percentile Rank por ser la mejor de sus dos variantes, con un (t)-statistic de 4.2516. Aunque su rendimiento baseline es inferior al de los modelos seleccionados anteriormente, mantiene evidencia de capacidad predictiva y permitirá comprobar empíricamente si la optimización de hiperparámetros modifica su posición relativa frente a los modelos lineales y LightGBM.

## 8. Hyperparameter Optimization

Con el objetivo de extraer el máximo potencial de las arquitecturas más competitivas sin incurrir en un gasto computacional ineficiente, en esta sección se somete a los modelos candidatos seleccionados a una fase de ajuste fino (fine-tuning). 
Esta etapa permite aislar y medir el Hyperparameter Gain —la ganancia neta en capacidad predictiva e $IR_{IC}$ atribuible exclusivamente a la optimización frente al modelo por defecto—. 

El proceso concluye con una evaluación consolidada que busca identificar la configuración final que mejor equilibre rendimiento cross-sectional, estabilidad temporal entre folds y robustez metodológica.

### 8.1 Hyperparameter Optimization Function

Para automatizar la búsqueda de la configuración óptima de los algoritmos se implementa una pipeline basada en Optuna (optimize_hyperparameters). 

A diferencia de las búsquedas tradicionales por malla (Grid Search), la función utiliza un muestreador bayesiano basado en estimadores de Parzen estructurados por árboles (TPE Sampler), optimizando de manera eficiente la exploración del espacio de parámetros.

En cada prueba (trial), el motor propone una combinación de hiperparámetros y ejecuta la evaluación sobre las particiones CPCV utilizando la pipeline de entrenamiento de la Sección 6.1. La función maximiza directamente el Rank IC medio out-of-fold, garantizando que los parámetros seleccionados prioricen la capacidad de ordenación cross-sectional del modelo. El proceso devuelve el estudio completo de optimización, el diccionario con los mejores hiperparámetros (best_params) y el rendimiento óptimo alcanzado, listos para transferirse al ajuste final.

### 8.2 Selection of Promising Models

Con base en la evaluación multicriterio y la lógica de selección establecida, las siguientes cinco configuraciones avanzan a la fase de optimización de hiperparámetros:

1. **Regresión Ridge (Percentile Rank)**: Líder absoluto del benchmark base ($\text{Rank IC} = 0.0343$).

2. **Regresión Lineal (Percentile Rank)**: Co-líder en capacidad de ordenación lineal, conservada como referencia base sin regularización para comparar el efecto de aplicar $L_2$ (Ridge) en la optimización.

3. **LightGBM (Z-Score)**: Modelo con mayor consistencia temporal, máximo Hit Rate ($58.40\%$) y superioridad en significancia ($t\text{-stat} = 6.1046$).

4. **LightGBM (Percentile Rank)**: Candidato no lineal de mayor magnitud en IC puro ($\text{Rank IC} = 0.0259$).

5. **XGBoost (Percentile Rank)**: Variante con mejor desempeño dentro de su familia, conservada para verificar si el ajuste de hiperparámetros logra rescatar su capacidad predictiva.


### 8.3 Optuna + CPCV

Para optimizar la carga computacional sin comprometer el rigor metodológico, la búsqueda de hiperparámetros con Optuna se ejecuta sobre un subconjunto representativo de 7 splits de CPCV (search_splits), extraídos mediante un muestreo aleatorio con semilla fija (SEED).

Esta decisión responde a que la etapa de tuning no precisa la estabilidad estadística absoluta de la evaluación final, sino una señal ordinal suficiente para que el algoritmo compare y ordene distintas configuraciones. Para mantener una comparativa totalmente justa y neutra entre arquitecturas, se reutiliza exactamente el mismo subconjunto de folds en la optimización de todos los candidatos. Finalmente, los mejores hiperparámetros hallados (best_params) se re-evalúan sobre la totalidad de los 21 folds de CPCV, garantizando que las métricas definitivas del proyecto conserven la máxima robustez fuera de muestra y carezcan de sesgos por subsamblado.

In [35]:
# =============================================================================
# Reduced CPCV splits for hyperparameter search
# =============================================================================

import random

random.seed(SEED)

N_SEARCH_SPLITS = 7

search_splits = random.sample(
    splits,
    k=N_SEARCH_SPLITS,
)

#### 8.3.1 Ridge

In [ ]:
# -----------------------------------------------------------------------------
# Hyperparameter Search Space
# -----------------------------------------------------------------------------

def ridge_parameter_space(trial):

    return {
        "alpha": trial.suggest_float(
            "alpha",
            1e-3,
            1e3,
            log=True,
        ),
        "random_state": SEED,
    }


# -----------------------------------------------------------------------------
# Hyperparameter Optimization
# -----------------------------------------------------------------------------

study_ridge_rank, best_params_ridge_rank, best_score_ridge_rank = (
    optimize_hyperparameters(
        model_cls=Ridge,
        parameter_space=ridge_parameter_space,
        X=X_rank,
        y=y_rank,
        splits=search_splits,
        n_trials=10,
        scoring_metric="Rank IC Mean",
        direction="maximize",
        random_state=SEED,
    )
)


# =============================================================================
# Optimization Results
# =============================================================================

print("\n")
print("=" * 80)
print("RIDGE REGRESSION — HYPERPARAMETER OPTIMIZATION")
print("=" * 80)

print("\nRepresentation: Percentile Rank")

print("\n" + "-" * 80)
print("BEST HYPERPARAMETERS")
print("-" * 80)

for parameter, value in best_params_ridge_rank.items():
    print(f"{parameter}: {value}")

print("\n" + "-" * 80)
print("BEST OPTIMIZATION SCORE")
print("-" * 80)

print(f"Rank IC Mean: {best_score_ridge_rank:.6f}")

print("\n" + "=" * 80)
print("END OF RIDGE HYPERPARAMETER OPTIMIZATION")
print("=" * 80)

Best trial: 1. Best value: 0.0310117: 100%|██████████| 10/10 [00:48<00:00,  4.90s/it]



RIDGE REGRESSION — HYPERPARAMETER OPTIMIZATION

Representation: Percentile Rank

--------------------------------------------------------------------------------
BEST HYPERPARAMETERS
--------------------------------------------------------------------------------
alpha: 506.1576888752306

--------------------------------------------------------------------------------
BEST OPTIMIZATION SCORE
--------------------------------------------------------------------------------
Rank IC Mean: 0.031012

END OF RIDGE HYPERPARAMETER OPTIMIZATION


#### 8.3.2 LightGBM — Z-Score

En esta fase se optimiza LightGBM sobre el conjunto **Z-Score** mediante **Optuna + CPCV**. Se define un espacio de búsqueda para los principales hiperparámetros que controlan la capacidad y regularización del modelo (`learning_rate`, `num_leaves`, `max_depth`, `min_child_samples`, `subsample` y `colsample_bytree`), mientras que `n_estimators=100` se mantiene fijo inicialmente para limitar el coste computacional. Optuna explora este espacio mediante el **TPE Sampler** y selecciona la configuración que maximiza el **Mean Rank IC** obtenido mediante CPCV. Se utilizarán inicialmente **20 trials**, ampliables si se observa un potencial de mejora significativo.


In [ ]:
# -----------------------------------------------------------------------------
# Hyperparameter Search Space
# -----------------------------------------------------------------------------

def lightgbm_parameter_space(trial):

    return {
        "n_estimators": 100,
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True,
        ),
        "num_leaves": trial.suggest_int(
            "num_leaves",
            15,
            127,
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            12,
        ),
        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            10,
            100,
        ),
        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0,
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0,
        ),
        "random_state": SEED,
        "n_jobs": 1,
        "verbosity": -1,
    }


# -----------------------------------------------------------------------------
# Hyperparameter Optimization
# -----------------------------------------------------------------------------

study_lgbm_z, best_params_lgbm_z, best_score_lgbm_z = (
    optimize_hyperparameters(
        model_cls=LGBMRegressor,
        parameter_space=lightgbm_parameter_space,
        X=X_z,
        y=y_z,
        splits=search_splits,
        n_trials=20,
        scoring_metric="Rank IC Mean",
        direction="maximize",
        random_state=SEED,
    )
)


# =============================================================================
# Optimization Results
# =============================================================================

print("\n")
print("=" * 80)
print("LIGHTGBM — HYPERPARAMETER OPTIMIZATION")
print("=" * 80)

print("\nRepresentation: Z-Score")

print("\n" + "-" * 80)
print("BEST HYPERPARAMETERS")
print("-" * 80)

for parameter, value in best_params_lgbm_z.items():
    print(f"{parameter}: {value}")

print("\n" + "-" * 80)
print("BEST OPTIMIZATION SCORE")
print("-" * 80)

print(f"Rank IC Mean: {best_score_lgbm_z:.6f}")

print("\n" + "=" * 80)
print("END OF LIGHTGBM Z-SCORE OPTIMIZATION")
print("=" * 80)


Best trial: 15. Best value: 0.0282588: 100%|██████████| 20/20 [08:21<00:00, 25.07s/it]



LIGHTGBM — HYPERPARAMETER OPTIMIZATION

Representation: Z-Score

--------------------------------------------------------------------------------
BEST HYPERPARAMETERS
--------------------------------------------------------------------------------
learning_rate: 0.0636301721905459
num_leaves: 87
max_depth: 3
min_child_samples: 89
subsample: 0.8043418314701809
colsample_bytree: 0.9399734292771992

--------------------------------------------------------------------------------
BEST OPTIMIZATION SCORE
--------------------------------------------------------------------------------
Rank IC Mean: 0.028259

END OF LIGHTGBM Z-SCORE OPTIMIZATION


#### 8.3.3 LightGBM — Percentile Rank

Se replica el procedimiento aplicado en LightGBM — Z-Score, utilizando ahora la representación Percentile Rank. Se mantiene el mismo espacio de búsqueda, los mismos 7 search_splits y 20 trials, garantizando así una comparación consistente entre ambas representaciones.

In [80]:
# -----------------------------------------------------------------------------
# Hyperparameter Optimization
# -----------------------------------------------------------------------------

study_lgbm_rank, best_params_lgbm_rank, best_score_lgbm_rank = (
    optimize_hyperparameters(
        model_cls=LGBMRegressor,
        parameter_space=lightgbm_parameter_space,
        X=X_rank,
        y=y_rank,
        splits=search_splits,
        n_trials=20,
        scoring_metric="Rank IC Mean",
        direction="maximize",
        random_state=SEED,
    )
)


# =============================================================================
# Optimization Results
# =============================================================================

print("\n")
print("=" * 80)
print("LIGHTGBM — HYPERPARAMETER OPTIMIZATION")
print("=" * 80)

print("\nRepresentation: Percentile Rank")

print("\n" + "-" * 80)
print("BEST HYPERPARAMETERS")
print("-" * 80)

for parameter, value in best_params_lgbm_rank.items():
    print(f"{parameter}: {value}")

print("\n" + "-" * 80)
print("BEST OPTIMIZATION SCORE")
print("-" * 80)

print(f"Rank IC Mean: {best_score_lgbm_rank:.6f}")

print("\n" + "=" * 80)
print("END OF LIGHTGBM PERCENTILE RANK OPTIMIZATION")
print("=" * 80)


Best trial: 15. Best value: 0.0296248: 100%|██████████| 20/20 [08:41<00:00, 26.08s/it]



LIGHTGBM — HYPERPARAMETER OPTIMIZATION

Representation: Percentile Rank

--------------------------------------------------------------------------------
BEST HYPERPARAMETERS
--------------------------------------------------------------------------------
learning_rate: 0.05423283594316056
num_leaves: 79
max_depth: 3
min_child_samples: 42
subsample: 0.7793524041158101
colsample_bytree: 0.8522169426320582

--------------------------------------------------------------------------------
BEST OPTIMIZATION SCORE
--------------------------------------------------------------------------------
Rank IC Mean: 0.029625

END OF LIGHTGBM PERCENTILE RANK OPTIMIZATION


#### 8.3.4 XGBoost — Percentile Rank

En esta fase se optimiza **XGBoost** sobre la representación **Percentile Rank** mediante Optuna + CPCV. El espacio de búsqueda incluye `learning_rate`, `max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`, `gamma`, `reg_alpha` y `reg_lambda`, cubriendo los principales parámetros de **capacidad, muestreo y regularización** del modelo. `n_estimators=100` se mantiene fijo inicialmente para controlar el coste computacional. Se utilizan los mismos **7 `search_splits` y 20 trials**, y Optuna selecciona la configuración que maximiza el **Mean Rank IC**.

In [ ]:
# -----------------------------------------------------------------------------
# Hyperparameter Search Space
# -----------------------------------------------------------------------------

def xgboost_parameter_space(trial):

    return {
        "n_estimators": 100,
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True,
        ),
        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10,
        ),
        "min_child_weight": trial.suggest_float(
            "min_child_weight",
            1,
            10,
        ),
        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0,
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0,
        ),
        "gamma": trial.suggest_float(
            "gamma",
            0,
            5,
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-4,
            10,
            log=True,
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-3,
            10,
            log=True,
        ),
        "random_state": SEED,
        "n_jobs": 1,
        "verbosity": 0,
    }


# =============================================================================
# Hyperparameter Optimization
# =============================================================================

study_xgb_rank, best_params_xgb_rank, best_score_xgb_rank = (
    optimize_hyperparameters(
        model_cls=XGBRegressor,
        parameter_space=xgboost_parameter_space,
        X=X_rank,
        y=y_rank,
        splits=search_splits,
        n_trials=20,
        scoring_metric="Rank IC Mean",
        direction="maximize",
        random_state=SEED,
    )
)


# =============================================================================
# Optimization Results
# =============================================================================

print("\n")
print("=" * 80)
print("XGBOOST — HYPERPARAMETER OPTIMIZATION")
print("=" * 80)

print("\nRepresentation: Percentile Rank")

print("\n" + "-" * 80)
print("BEST HYPERPARAMETERS")
print("-" * 80)

for parameter, value in best_params_xgb_rank.items():
    print(f"{parameter}: {value}")

print("\n" + "-" * 80)
print("BEST OPTIMIZATION SCORE")
print("-" * 80)

print(f"Rank IC Mean: {best_score_xgb_rank:.6f}")

print("\n" + "=" * 80)
print("END OF XGBOOST PERCENTILE RANK OPTIMIZATION")
print("=" * 80)


Best trial: 6. Best value: 0.030095: 100%|██████████| 20/20 [10:31<00:00, 31.56s/it]



XGBOOST — HYPERPARAMETER OPTIMIZATION

Representation: Percentile Rank

--------------------------------------------------------------------------------
BEST HYPERPARAMETERS
--------------------------------------------------------------------------------
learning_rate: 0.05143828405076928
max_depth: 4
min_child_weight: 9.726261649881026
subsample: 0.9100531293444458
colsample_bytree: 0.9757995766256756
gamma: 4.474136752138244
reg_alpha: 0.09761125443110447
reg_lambda: 4.869640941520899

--------------------------------------------------------------------------------
BEST OPTIMIZATION SCORE
--------------------------------------------------------------------------------
Rank IC Mean: 0.030095

END OF XGBOOST PERCENTILE RANK OPTIMIZATION


### 8.4 Optimized Model Evaluation


#### 8.4.1 Re-training with Optimal Hyperparameters

Una vez finalizada la búsqueda de hiperparámetros, las configuraciones óptimas seleccionadas por Optuna se evalúan nuevamente utilizando las 21 particiones CPCV completas. De este modo, las métricas obtenidas en esta fase constituyen la evaluación robusta y comparable de los modelos optimizados, independiente del subconjunto de 7 folds utilizado durante la búsqueda.


In [16]:
# -----------------------------------------------------------------------------
# Ridge — Percentile Rank
# -----------------------------------------------------------------------------

oof_preds_ridge_rank_opt, fold_metrics_ridge_rank_opt, agg_metrics_ridge_rank_opt = (
    run_cpcv_training(
        model_cls=Ridge,
        model_params=best_params_ridge_rank,
        X=X_rank,
        y=y_rank,
        splits=splits,
        rank_method="spearman",
    )
)


In [17]:
# -----------------------------------------------------------------------------
# Light GBM — Z-score
# -----------------------------------------------------------------------------

oof_preds_lgbm_z_opt, fold_metrics_lgbm_z_opt, agg_metrics_lgbm_z_opt = (
    run_cpcv_training(
        model_cls=LGBMRegressor,
        model_params=best_params_lgbm_z,
        X=X_z,
        y=y_z,
        splits=splits,
        rank_method="spearman",
    )
)

In [18]:
# -----------------------------------------------------------------------------
# Light GBM — Percentile Rank
# -----------------------------------------------------------------------------

oof_preds_lgbm_rank_opt, fold_metrics_lgbm_rank_opt, agg_metrics_lgbm_rank_opt = (
    run_cpcv_training(
        model_cls=LGBMRegressor,
        model_params=best_params_lgbm_rank,
        X=X_rank,
        y=y_rank,
        splits=splits,
        rank_method="spearman",
    )
)

In [19]:
# -----------------------------------------------------------------------------
# XGBoost — Percentile Rank
# -----------------------------------------------------------------------------

oof_preds_xgb_rank_opt, fold_metrics_xgb_rank_opt, agg_metrics_xgb_rank_opt = (
    run_cpcv_training(
        model_cls=XGBRegressor,
        model_params=best_params_xgb_rank,
        X=X_rank,
        y=y_rank,
        splits=splits,
        rank_method="spearman",
    )
)

#### 8.4.2 Baseline vs Optimized Performance

Para evaluar el impacto real del ajuste de hiperparámetros, calculamos la variación neta de cada métrica como $\Delta = \text{Optimizado} - \text{Baseline}$. Bajo este enfoque, las variaciones positivas en la capacidad de ordenación, consistencia y precisión ($IC$, $IR$, $Hit Rate$, $t\text{-stat}$) y las variaciones negativas en las métricas de error ($RMSE$, $MA E$) reflejan una mejora efectiva tras el tuning.

In [ ]:
comparisons = [
    {
        "Model": "Ridge",
        "Scaling / Representation": "Percentile Rank",
        "Baseline": agg_metrics_ridge_rank,
        "Optimized": agg_metrics_ridge_rank_opt,
    },
    {
        "Model": "LightGBM",
        "Scaling / Representation": "Z-Score",
        "Baseline": agg_metrics_lgbm_z,
        "Optimized": agg_metrics_lgbm_z_opt,
    },
    {
        "Model": "LightGBM",
        "Scaling / Representation": "Percentile Rank",
        "Baseline": agg_metrics_lgbm_rank,
        "Optimized": agg_metrics_lgbm_rank_opt,
    },
    {
        "Model": "XGBoost",
        "Scaling / Representation": "Percentile Rank",
        "Baseline": agg_metrics_xgb_rank,
        "Optimized": agg_metrics_xgb_rank_opt,
    },
]


# -----------------------------------------------------------------------------
# Build delta table
# -----------------------------------------------------------------------------

rows = []

for comparison in comparisons:

    baseline = extract_mean_metrics(comparison["Baseline"])
    optimized = extract_mean_metrics(comparison["Optimized"])

    row = {
        "Model": comparison["Model"],
        "Scaling / Representation": comparison["Scaling / Representation"],
    }

    # Δ = Optimized - Baseline
    row["Δ Rank IC Mean"] = (
        optimized["Rank IC Mean"] - baseline["Rank IC Mean"]
    )

    row["Δ Rank IC IR"] = (
        optimized["Rank IC Information Ratio"]
        - baseline["Rank IC Information Ratio"]
    )

    row["Δ Hit Rate (%)"] = (
        optimized["Rank IC Hit Rate (%)"]
        - baseline["Rank IC Hit Rate (%)"]
    )

    row["Δ t-statistic"] = (
        optimized["Rank IC t-statistic"]
        - baseline["Rank IC t-statistic"]
    )

    row["Δ RMSE"] = (
        optimized["RMSE"] - baseline["RMSE"]
    )

    row["Δ MAE"] = (
        optimized["MAE"] - baseline["MAE"]
    )

    rows.append(row)


baseline_vs_optimized_delta = pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

display(
    baseline_vs_optimized_delta.style
    .hide(axis="index")
)


Model,Scaling / Representation,Δ Rank IC Mean,Δ Rank IC IR,Δ Hit Rate (%),Δ t-statistic,Δ RMSE,Δ MAE
Ridge,Percentile Rank,0.000007,0.000047,-0.004762,0.001472,-0.000000,-0.000001
LightGBM,Z-Score,0.004500,0.010835,0.228571,0.342648,-0.000615,-0.000278
LightGBM,Percentile Rank,0.004395,0.010351,0.123810,0.327339,-0.000594,-0.000236
XGBoost,Percentile Rank,0.020648,0.104693,3.485714,3.310692,-0.001884,-0.000983


La comparación entre los modelos optimizados y sus configuraciones baseline ($\Delta = \text{Optimizado} - \text{Baseline}$) muestra comportamientos heterogéneos según la arquitectura.

En la Regresión Ridge, el ajuste de hiperparámetros no altera de forma apreciable su desempeño, manteniendo variaciones prácticamente nulas en todas las métricas ($\Delta \text{Rank IC} = +0.000007$, $\Delta \text{Hit Rate} = -0.0048\%$).

En la familia LightGBM, ambas representaciones experimentan mejoras moderadas y homogéneas tras el tuning. La variante bajo Z-Score incrementa su Rank IC en $+0.0045$, su Information Ratio en $+0.0108$, su Hit Rate en $+0.2286\%$ y su $t$-statistic en $+0.3426$, reduciendo simultáneamente el RMSE en $-0.0006$. Por su parte, la versión en Percentile Rank registra incrementos similares, destacando un aumento de $+0.0044$ en Rank IC y de $+0.3273$ en el $t$-statistic.

Por último, XGBoost (Percentile Rank) refleja los mayores incrementos absolutos del panel. Tras la optimización, su Rank IC medio aumenta en $+0.0206$, el Information Ratio sube $+0.1047$, la tasa de aciertos asciende un $+3.4857\%$ y el $t$-statistic experimenta un incremento de $+3.3107$, acompañado de reducciones de $-0.0019$ en RMSE y $-0.0010$ en MAE.


#### 8.4.3 Optimized Hyperparameters

La siguiente tabla resume los hiperparámetros óptimos obtenidos durante la búsqueda con Optuna para cada modelo y representación. Se presenta cada hiperparámetro en una fila independiente para facilitar su lectura, comparación y posterior consulta. La tabla se almacena en formato Parquet dentro de data/model_results/ para poder reutilizarla sin necesidad de repetir el proceso de optimización.

In [ ]:
import json
# -----------------------------------------------------------------------------
# Collect optimized hyperparameters
# -----------------------------------------------------------------------------

optimized_hyperparameters = {
    "Ridge": {
        "Percentile Rank": best_params_ridge_rank,
    },
    "LightGBM": {
        "Z-Score": best_params_lgbm_z,
        "Percentile Rank": best_params_lgbm_rank,
    },
    "XGBoost": {
        "Percentile Rank": best_params_xgb_rank,
    },
}


# -----------------------------------------------------------------------------
# Build summary table
# -----------------------------------------------------------------------------

hyperparameter_rows = []

for model, representations in optimized_hyperparameters.items():

    for representation, params in representations.items():

        for parameter, value in params.items():

            hyperparameter_rows.append({
                "Model": model,
                "Scaling / Representation": representation,
                "Hyperparameter": parameter,
                "Optimal Value": value,
            })


optimized_hyperparameters_table = pd.DataFrame(
    hyperparameter_rows
)

# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

pd.set_option("display.float_format", "{:.6f}".format)

optimized_hyperparameters_table.to_parquet("../data/model_results/optimized_hyperparameters_table.parquet")


Model,Scaling / Representation,Hyperparameter,Optimal Value
Ridge,Percentile Rank,alpha,506.157689
LightGBM,Z-Score,learning_rate,0.063630
LightGBM,Z-Score,num_leaves,87.000000
LightGBM,Z-Score,max_depth,3.000000
LightGBM,Z-Score,min_child_samples,89.000000
LightGBM,Z-Score,subsample,0.804342
LightGBM,Z-Score,colsample_bytree,0.939973
LightGBM,Percentile Rank,learning_rate,0.054233
LightGBM,Percentile Rank,num_leaves,79.000000
LightGBM,Percentile Rank,max_depth,3.000000


Además de la tabla de consulta, las configuraciones óptimas se almacenan en formato JSON, conservando la estructura jerárquica de modelo, representación e hiperparámetros. Este archivo permite mantener un registro reproducible de las configuraciones seleccionadas por Optuna y utilizarlas posteriormente sin necesidad de volver a ejecutar la búsqueda.

In [97]:
import json
from pathlib import Path

# =============================================================================
# Save Optimized Hyperparameters
# =============================================================================

output_path = Path(
    r"C:\Users\dppec\OneDrive\Escritorio\PROYECTOS DE PYTHON"
    r"\quant-portfolio-project\data\model_results"
    r"\optimized_hyperparameters.json"
)

# Create directory if it does not exist
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as file:
    json.dump(
        optimized_hyperparameters,
        file,
        indent=4,
    )

print(f"Optimized hyperparameters saved to: {output_path}")

Optimized hyperparameters saved to: C:\Users\dppec\OneDrive\Escritorio\PROYECTOS DE PYTHON\quant-portfolio-project\data\model_results\optimized_hyperparameters.json


### 8.5 Final Ranking After Optimization

####  8.5.1 Construction of the Optimized Ranking

Para la elaboración del ranking final de la Sección 8, la evaluación se centra exclusivamente en las versiones tuneadas de los cuatro candidatos sometidos a la optimización con Optuna (Ridge Percentile Rank, LightGBM Z-Score, LightGBM Percentile Rank y XGBoost Percentile Rank), ya que en todos los casos la búsqueda de hiperparámetros mejoró o igualó el desempeño de sus versiones baseline.

Asimismo, se reincorpora al panel la Regresión Lineal (Percentile Rank) sin ajustar para mantener la referencia directa de la señal lineal pura sin regularización dentro del conjunto de modelos evaluados.

In [101]:
# -----------------------------------------------------------------------------
# 1. Build optimized benchmark dataset
# -----------------------------------------------------------------------------

optimized_ranking_data = [
    {
        "Model": "Linear Regression",
        "Scaling / Representation": "Percentile Rank",
        "Metrics": extract_mean_metrics(agg_metrics_lr_rank),
    },
    {
        "Model": "Ridge",
        "Scaling / Representation": "Percentile Rank",
        "Metrics": extract_mean_metrics(agg_metrics_ridge_rank_opt),
    },
    {
        "Model": "LightGBM",
        "Scaling / Representation": "Z-Score",
        "Metrics": extract_mean_metrics(agg_metrics_lgbm_z_opt),
    },
    {
        "Model": "LightGBM",
        "Scaling / Representation": "Percentile Rank",
        "Metrics": extract_mean_metrics(agg_metrics_lgbm_rank_opt),
    },
    {
        "Model": "XGBoost",
        "Scaling / Representation": "Percentile Rank",
        "Metrics": extract_mean_metrics(agg_metrics_xgb_rank_opt),
    },
]


# -----------------------------------------------------------------------------
# 2. Create DataFrame
# -----------------------------------------------------------------------------

rows = []

for item in optimized_ranking_data:

    metrics = item["Metrics"]

    rows.append({
        "Model": item["Model"],
        "Scaling / Representation": item["Scaling / Representation"],
        "Rank IC Mean": metrics["Rank IC Mean"],
        "Rank IC Information Ratio": metrics["Rank IC Information Ratio"],
        "Rank IC Hit Rate (%)": metrics["Rank IC Hit Rate (%)"],
        "Rank IC t-statistic": metrics["Rank IC t-statistic"],
    })


df_optimized_benchmark = pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 3. Convert performance metrics into percentile ranks
# -----------------------------------------------------------------------------

df_score_optimized = df_optimized_benchmark.copy()

df_score_optimized["Rank IC Mean Score"] = (
    df_score_optimized["Rank IC Mean"]
    .rank(method="average", pct=True)
)

df_score_optimized["Rank IC IR Score"] = (
    df_score_optimized["Rank IC Information Ratio"]
    .rank(method="average", pct=True)
)

df_score_optimized["Hit Rate Score"] = (
    df_score_optimized["Rank IC Hit Rate (%)"]
    .rank(method="average", pct=True)
)


# -----------------------------------------------------------------------------
# 4. Calculate weighted multicriteria score
# -----------------------------------------------------------------------------

df_score_optimized["Multicriteria Score"] = (
    0.50 * df_score_optimized["Rank IC Mean Score"]
    + 0.30 * df_score_optimized["Rank IC IR Score"]
    + 0.20 * df_score_optimized["Hit Rate Score"]
)


# -----------------------------------------------------------------------------
# 5. Create final ranking
# -----------------------------------------------------------------------------

df_score_optimized = (
    df_score_optimized
    .sort_values(
        "Multicriteria Score",
        ascending=False
    )
    .reset_index(drop=True)
)

df_score_optimized["Position"] = (
    df_score_optimized["Multicriteria Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -----------------------------------------------------------------------------
# 6. Select columns for the ranking
# -----------------------------------------------------------------------------

df_model_ranking_optimized = df_score_optimized[
    [
        "Position",
        "Model",
        "Scaling / Representation",
        "Rank IC Mean",
        "Rank IC Information Ratio",
        "Rank IC Hit Rate (%)",
        "Rank IC t-statistic",
        "Multicriteria Score",
    ]
].copy()


# -----------------------------------------------------------------------------
# 7. Round numerical values
# -----------------------------------------------------------------------------

numeric_columns = (
    df_model_ranking_optimized
    .select_dtypes(include="number")
    .columns
)

df_model_ranking_optimized[numeric_columns] = (
    df_model_ranking_optimized[numeric_columns]
    .round(4)
)


# =============================================================================
# Display Results
# =============================================================================

print("\n" + "=" * 110)
print("8.5.1 OPTIMIZED MULTICRITERIA MODEL RANKING")
print("=" * 110)

print("\nScore composition:")
print("  Rank IC Mean              → 50%")
print("  Rank IC Information Ratio → 30%")
print("  Rank IC Hit Rate          → 20%")

print("\nPercentile ranking:")
print("  Higher values = better performance")

print("\n" + "-" * 110)

display(
    df_model_ranking_optimized.style
    .hide(axis="index")
)


8.5.1 OPTIMIZED MULTICRITERIA MODEL RANKING

Score composition:
  Rank IC Mean              → 50%
  Rank IC Information Ratio → 30%
  Rank IC Hit Rate          → 20%

Percentile ranking:
  Higher values = better performance

--------------------------------------------------------------------------------------------------------------


Position,Model,Scaling / Representation,Rank IC Mean,Rank IC Information Ratio,Rank IC Hit Rate (%),Rank IC t-statistic,Multicriteria Score
1,XGBoost,Percentile Rank,0.032900,0.239100,58.428600,7.562300,0.760000
2,Ridge,Percentile Rank,0.034300,0.182800,56.947600,5.779100,0.660000
3,Linear Regression,Percentile Rank,0.034300,0.182700,56.952400,5.777600,0.540000
3,LightGBM,Z-Score,0.029400,0.203900,58.628600,6.447300,0.540000
5,LightGBM,Percentile Rank,0.030300,0.196800,58.028600,6.222900,0.500000


La aplicación del mismo criterio multicriterio utilizado en el benchmark baseline permite obtener un nuevo orden relativo de los modelos tras incorporar las configuraciones seleccionadas mediante la optimización de hiperparámetros. El ranking resultante sitúa a **XGBoost (Percentile Rank)** en primera posición, con un `Multicriteria Score` de **0.7600**, seguido de **Ridge (Percentile Rank)** con **0.6600**. Ambos modelos presentan los valores más elevados del Score global, aunque muestran perfiles diferentes en las métricas que lo componen.

**XGBoost (Percentile Rank)** registra un `Rank IC Mean` de **0.0329**, un `Rank IC Information Ratio` de **0.2391**, un `Hit Rate` de **58.43%** y un `t-statistic` de **7.5623**. Destaca especialmente por sus valores de Information Ratio y significancia estadística, que son los más elevados del conjunto de modelos evaluados en esta fase.

**Ridge (Percentile Rank)** ocupa la segunda posición con un `Rank IC Mean` de **0.0343**, el valor más elevado del panel, acompañado de un `Information Ratio` de **0.1828**, un `Hit Rate` de **56.95%** y un `t-statistic` de **5.7791**. Su perfil muestra una mayor fuerza media de ordenación, aunque valores inferiores en las métricas asociadas a consistencia y significancia respecto a XGBoost.

La **Regresión Lineal (Percentile Rank)** aparece en tercera posición con un `Multicriteria Score` de **0.5400**. Sus métricas son prácticamente idénticas a las de Ridge, con un `Rank IC Mean` de **0.0343**, un `Information Ratio` de **0.1827**, un `Hit Rate` de **56.95%** y un `t-statistic` de **5.7776**. Esta proximidad refleja el comportamiento muy similar de ambas arquitecturas bajo esta representación.

También con un `Score` de **0.5400`**, **LightGBM (Z-Score)** comparte la tercera posición. Su `Rank IC Mean`es de **0.0294**, mientras que alcanza un`Information Ratio`de **0.2039**, un`Hit Rate`de **58.63%** y un`t-statistic` de **6.4473**. De este modo, presenta un perfil caracterizado por una menor fuerza media de ordenación, pero valores elevados en las métricas de consistencia y acierto direccional.

Por último, **LightGBM (Percentile Rank)** ocupa la quinta posición, con un `Multicriteria Score` de **0.5000**. Esta configuración registra un `Rank IC Mean` de **0.0303**, un `Information Ratio` de **0.1968**, un `Hit Rate` de **58.03%** y un `t-statistic` de **6.2229**.

En conjunto, el ranking optimizado presenta una distribución diferenciada de perfiles predictivos: mientras que las configuraciones lineales muestran los mayores valores de `Rank IC Mean`, las configuraciones basadas en *gradient boosting* presentan valores superiores en varias de las métricas asociadas a consistencia temporal, acierto direccional y significancia estadística. Estas diferencias permiten establecer la base cuantitativa para el análisis individual de los modelos en las siguientes secciones.


#### 8.6 Random Forest — Percentile Rank

Como análisis adicional de robustez arquitectónica, se incorpora **Random Forest** sobre la representación Percentile Rank. Dado que los resultados anteriores muestran que la optimización de hiperparámetros puede modificar sustancialmente el desempeño de los modelos basados en árboles, se omite una evaluación baseline independiente y se procede directamente a la optimización mediante **Optuna + CPCV**. 

Dado el elevado coste computacional de Random Forest sobre el dataset completo (~1.6M filas), la búsqueda se ejecutó sobre un subconjunto reducido de folds de CPCV (7 de los 21 originales, los mismos usados para XGBoost y LightGBM, para mantener la comparación entre modelos consistente). Se fijó n_jobs=4 en el estimador, aprovechando que en Random Forest los árboles se construyen de forma independiente (bagging) y por tanto paralelizan sin la penalización de contención observada en modelos de boosting secuencial como LightGBM.

En cuanto al espacio de búsqueda, max_depth se acotó a un rango bajo (3-6) y se añadió max_samples (fracción de bootstrap por árbol, 0.2-0.5) para reducir el coste de cada split individual — relevante porque, a diferencia de XGBoost/LightGBM (que usan histogramas), sklearn's Random Forest evalúa splits de forma exacta, lo que lo hace considerablemente más lento por árbol a este volumen de datos. Esta decisión no es solo una medida de eficiencia: dado que el hallazgo previo con XGBoost mostró que el modelo óptimo tiende a comportarse de forma casi-lineal bajo fuerte regularización, acotar la profundidad y el bootstrap de RF hacia el mismo rango de regularización mantiene la comparación "justa" entre familias de árboles, en vez de permitir que unos modelos exploren capacidades mucho mayores que otros por simple disponibilidad de tiempo de cómputo. Se comenzó con un presupuesto exploratorio de 5 trials para validar si RF muestra señal prometedora antes de justificar una búsqueda más extensa.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# =============================================================================
# Random Forest — Hyperparameter Search Space
# =============================================================================

def random_forest_parameter_space(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 15),
        "max_features": trial.suggest_float("max_features", 0.5, 1.0),
        "max_samples": trial.suggest_float("max_samples", 0.2, 0.5),
        "random_state": SEED,
        "n_jobs": 4,
    }
# =============================================================================
# Random Forest — Hyperparameter Optimization
# =============================================================================

study_rf_rank, best_params_rf_rank, best_score_rf_rank = (
    optimize_hyperparameters(  # la función original, sin tocar
        model_cls=RandomForestRegressor,
        parameter_space=random_forest_parameter_space,
        X=X_rank,
        y=y_rank,
        splits=search_splits,
        n_trials=20,
        scoring_metric="Rank IC Mean",
        direction="maximize",
        random_state=SEED,
    )
)

# =============================================================================
# Optimization Results
# =============================================================================

print("\n")
print("=" * 80)
print("RANDOM FOREST — HYPERPARAMETER OPTIMIZATION")
print("=" * 80)

print("\nRepresentation: Percentile Rank")

print("\n" + "-" * 80)
print("BEST HYPERPARAMETERS")
print("-" * 80)

for parameter, value in best_params_rf_rank.items():

    if isinstance(value, float):
        print(f"{parameter:<20}: {value:.6f}")
    else:
        print(f"{parameter:<20}: {value}")

print("\n" + "-" * 80)
print("BEST OPTIMIZATION SCORE")
print("-" * 80)

print(f"Rank IC Mean: {best_score_rf_rank:.6f}")

print("\n" + "=" * 80)
print("END OF RANDOM FOREST OPTIMIZATION")
print("=" * 80)

Best trial: 14. Best value: 0.0309523: 100%|██████████| 20/20 [39:22<00:00, 118.14s/it]



RANDOM FOREST — HYPERPARAMETER OPTIMIZATION

Representation: Percentile Rank

--------------------------------------------------------------------------------
BEST HYPERPARAMETERS
--------------------------------------------------------------------------------
n_estimators        : 100
max_depth           : 3
min_samples_leaf    : 5
max_features        : 0.891728
max_samples         : 0.424962

--------------------------------------------------------------------------------
BEST OPTIMIZATION SCORE
--------------------------------------------------------------------------------
Rank IC Mean: 0.030952

END OF RANDOM FOREST OPTIMIZATION


In [25]:
# =============================================================================
# Random Forest — Final CPCV Evaluation
# =============================================================================

# Add back the fixed parameters that Optuna's best_params doesn't include
best_params_rf_rank["random_state"] = SEED
best_params_rf_rank["n_jobs"] = 4  # single run, safe to parallelize here

metrics_rf_rank, predictions_rf_rank, agg_metrics_rf_rank = (
    run_cpcv_training(
        model_cls=RandomForestRegressor,
        model_params=best_params_rf_rank,
        X=X_rank,
        y=y_rank,
        splits=splits,  # Full 21-fold CPCV
        rank_method="spearman",
    )
)

####  8.6.1 Construction of the Optimized Ranking with Random Forest

Volvemos a constuir la misma tabla con los resultados de los modelos optimizados, ahora incluyendo en el ranking el Random Forest. 

In [ ]:
# =============================================================================
# Recover optimized hyperparameters from saved table
# =============================================================================

optimized_hyperparameters_table = pd.read_parquet(
    "../data/model_results/optimized_hyperparameters_table.parquet"
)


def recover_hyperparameters(
    table,
    model,
    representation,
    integer_parameters=None,
):

    subset = table[
        (table["Model"] == model)
        & (
            table["Scaling / Representation"]
            == representation
        )
    ].copy()

    params = dict(
        zip(
            subset["Hyperparameter"],
            subset["Optimal Value"],
        )
    )

    # Convert integer-valued hyperparameters back to int
    if integer_parameters is not None:

        for parameter in integer_parameters:

            if parameter in params:
                params[parameter] = int(params[parameter])

    return params


# -----------------------------------------------------------------------------
# Recover model configurations
# -----------------------------------------------------------------------------

best_params_ridge_rank = recover_hyperparameters(
    optimized_hyperparameters_table,
    "Ridge",
    "Percentile Rank",
)


best_params_lgbm_z = recover_hyperparameters(
    optimized_hyperparameters_table,
    "LightGBM",
    "Z-Score",
    integer_parameters=[
        "n_estimators",
        "num_leaves",
        "max_depth",
        "min_child_samples",
    ],
)


best_params_lgbm_rank = recover_hyperparameters(
    optimized_hyperparameters_table,
    "LightGBM",
    "Percentile Rank",
    integer_parameters=[
        "n_estimators",
        "num_leaves",
        "max_depth",
        "min_child_samples",
    ],
)


best_params_xgb_rank = recover_hyperparameters(
    optimized_hyperparameters_table,
    "XGBoost",
    "Percentile Rank",
    integer_parameters=[
        "n_estimators",
        "max_depth",
    ],
)


best_params_rf_rank = recover_hyperparameters(
    optimized_hyperparameters_table,
    "Random Forest",
    "Percentile Rank",
    integer_parameters=[
        "n_estimators",
        "max_depth",
        "min_samples_leaf",
    ],
)

In [20]:
# =============================================================================
# Optimized Models — Full CPCV Evaluation
# =============================================================================


# -----------------------------------------------------------------------------
# Ridge — Percentile Rank
# -----------------------------------------------------------------------------

oof_preds_ridge_rank_opt, fold_metrics_ridge_rank_opt, agg_metrics_ridge_rank_opt = (
    run_cpcv_training(
        model_cls=Ridge,
        model_params=best_params_ridge_rank,
        X=X_rank,
        y=y_rank,
        splits=splits,
        rank_method="spearman",
    )
)


# -----------------------------------------------------------------------------
# LightGBM — Z-Score
# -----------------------------------------------------------------------------

oof_preds_lgbm_z_opt, fold_metrics_lgbm_z_opt, agg_metrics_lgbm_z_opt = (
    run_cpcv_training(
        model_cls=LGBMRegressor,
        model_params=best_params_lgbm_z,
        X=X_z,
        y=y_z,
        splits=splits,
        rank_method="spearman",
    )
)


# -----------------------------------------------------------------------------
# LightGBM — Percentile Rank
# -----------------------------------------------------------------------------

oof_preds_lgbm_rank_opt, fold_metrics_lgbm_rank_opt, agg_metrics_lgbm_rank_opt = (
    run_cpcv_training(
        model_cls=LGBMRegressor,
        model_params=best_params_lgbm_rank,
        X=X_rank,
        y=y_rank,
        splits=splits,
        rank_method="spearman",
    )
)


# -----------------------------------------------------------------------------
# XGBoost — Percentile Rank
# -----------------------------------------------------------------------------

oof_preds_xgb_rank_opt, fold_metrics_xgb_rank_opt, agg_metrics_xgb_rank_opt = (
    run_cpcv_training(
        model_cls=XGBRegressor,
        model_params=best_params_xgb_rank,
        X=X_rank,
        y=y_rank,
        splits=splits,
        rank_method="spearman",
    )
)


# -----------------------------------------------------------------------------
# Random Forest — Percentile Rank
# -----------------------------------------------------------------------------

oof_preds_rf_rank_opt, fold_metrics_rf_rank_opt, agg_metrics_rf_rank_opt = (
    run_cpcv_training(
        model_cls=RandomForestRegressor,
        model_params=best_params_rf_rank,
        X=X_rank,
        y=y_rank,
        splits=splits,
        rank_method="spearman",
    )
)

In [25]:
# -----------------------------------------------------------------------------
# 1. Build optimized benchmark dataset including RF
# -----------------------------------------------------------------------------

optimized_ranking_data = [
    {
        "Model": "Linear Regression",
        "Scaling / Representation": "Percentile Rank",
        "Metrics": extract_mean_metrics(agg_metrics_lr_rank),
    },
    {
        "Model": "Ridge",
        "Scaling / Representation": "Percentile Rank",
        "Metrics": extract_mean_metrics(agg_metrics_ridge_rank_opt),
    },
    {
        "Model": "LightGBM",
        "Scaling / Representation": "Z-Score",
        "Metrics": extract_mean_metrics(agg_metrics_lgbm_z_opt),
    },
    {
        "Model": "LightGBM",
        "Scaling / Representation": "Percentile Rank",
        "Metrics": extract_mean_metrics(agg_metrics_lgbm_rank_opt),
    },
    {
        "Model": "XGBoost",
        "Scaling / Representation": "Percentile Rank",
        "Metrics": extract_mean_metrics(agg_metrics_xgb_rank_opt),
    },
    {
        "Model": "Random Forest",
        "Scaling / Representation": "Percentile Rank",
        "Metrics": extract_mean_metrics(agg_metrics_rf_rank_opt),
    },
]

# -----------------------------------------------------------------------------
# 2. Create DataFrame
# -----------------------------------------------------------------------------

rows = []

for item in optimized_ranking_data:

    metrics = item["Metrics"]

    rows.append({
        "Model": item["Model"],
        "Scaling / Representation": item["Scaling / Representation"],
        "Rank IC Mean": metrics["Rank IC Mean"],
        "Rank IC Information Ratio": metrics["Rank IC Information Ratio"],
        "Rank IC Hit Rate (%)": metrics["Rank IC Hit Rate (%)"],
        "Rank IC t-statistic": metrics["Rank IC t-statistic"],
    })

df_optimized_benchmark = pd.DataFrame(rows)

# -----------------------------------------------------------------------------
# 3. Convert performance metrics into percentile ranks
# -----------------------------------------------------------------------------

df_score_optimized = df_optimized_benchmark.copy()

df_score_optimized["Rank IC Mean Score"] = (
    df_score_optimized["Rank IC Mean"]
    .rank(method="average", pct=True)
)

df_score_optimized["Rank IC IR Score"] = (
    df_score_optimized["Rank IC Information Ratio"]
    .rank(method="average", pct=True)
)

df_score_optimized["Hit Rate Score"] = (
    df_score_optimized["Rank IC Hit Rate (%)"]
    .rank(method="average", pct=True)
)

# -----------------------------------------------------------------------------
# 4. Calculate weighted multicriteria score
# -----------------------------------------------------------------------------

df_score_optimized["Multicriteria Score"] = (
    0.50 * df_score_optimized["Rank IC Mean Score"]
    + 0.30 * df_score_optimized["Rank IC IR Score"]
    + 0.20 * df_score_optimized["Hit Rate Score"]
)

# -----------------------------------------------------------------------------
# 5. Create final ranking
# -----------------------------------------------------------------------------

df_score_optimized = (
    df_score_optimized
    .sort_values(
        "Multicriteria Score",
        ascending=False
    )
    .reset_index(drop=True)
)

df_score_optimized["Position"] = (
    df_score_optimized["Multicriteria Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

# -----------------------------------------------------------------------------
# 6. Select columns for the ranking
# -----------------------------------------------------------------------------

df_model_ranking_optimized = df_score_optimized[
    [
        "Position",
        "Model",
        "Scaling / Representation",
        "Rank IC Mean",
        "Rank IC Information Ratio",
        "Rank IC Hit Rate (%)",
        "Rank IC t-statistic",
        "Multicriteria Score",
    ]
].copy()

# -----------------------------------------------------------------------------
# 7. Round numerical values
# -----------------------------------------------------------------------------

numeric_columns = (
    df_model_ranking_optimized
    .select_dtypes(include="number")
    .columns
)

df_model_ranking_optimized[numeric_columns] = (
    df_model_ranking_optimized[numeric_columns]
    .round(4)
)

# =============================================================================
# Display Results
# =============================================================================

print("\n" + "=" * 110)
print("8.5.1 OPTIMIZED MULTICRITERIA MODEL RANKING")
print("=" * 110)

print("\nScore composition:")
print("  Rank IC Mean              → 50%")
print("  Rank IC Information Ratio → 30%")
print("  Rank IC Hit Rate          → 20%")

print("\nPercentile ranking:")
print("  Higher values = better performance")

print("\n" + "-" * 110)

display(
    df_model_ranking_optimized.style
    .hide(axis="index")
)

df_model_ranking_optimized.to_parquet("../data/model_results/df_model_ranking_optimized.parquet")


8.5.1 OPTIMIZED MULTICRITERIA MODEL RANKING

Score composition:
  Rank IC Mean              → 50%
  Rank IC Information Ratio → 30%
  Rank IC Hit Rate          → 20%

Percentile ranking:
  Higher values = better performance

--------------------------------------------------------------------------------------------------------------


Position,Model,Scaling / Representation,Rank IC Mean,Rank IC Information Ratio,Rank IC Hit Rate (%),Rank IC t-statistic,Multicriteria Score
1,XGBoost,Percentile Rank,0.032900,0.239100,58.428600,7.562300,0.716700
2,Random Forest,Percentile Rank,0.033700,0.207900,57.857100,6.572800,0.683300
3,Ridge,Percentile Rank,0.034300,0.182800,56.947600,5.779100,0.633300
4,Linear Regression,Percentile Rank,0.034300,0.182700,56.952400,5.777600,0.533300
5,LightGBM,Z-Score,0.029400,0.203900,58.628600,6.447300,0.483300
6,LightGBM,Percentile Rank,0.030300,0.196800,58.028600,6.222900,0.450000


La incorporación de Random Forest (Percentile Rank) tras su optimización reordena el podio global, situándose directamente en la segunda posición con un Score de 0.6833. El algoritmo demuestra un equilibrio notable entre fuerza bruta y estabilidad: registra el segundo Rank IC Mean más alto del panel (0.0337), superando a XGBoost en capacidad pura de ordenación y quedando solo por detrás de la referencia lineal (Ridge, 0.0343). 

Asimismo, alcanza una gran consistencia y significancia estadística (IR de 0.2079, Hit Rate del 57.86% y un $t$-stat de 6.5728), superando holgadamente a LightGBM y consolidándose como la segunda arquitectura basada en árboles más competitiva del estudio.

####  8.6.2 Save to parquet

In [44]:
import json

# -----------------------------------------------------------------------------
# Load existing hyperparameters
# -----------------------------------------------------------------------------

json_path = Path(
    r"C:\Users\dppec\OneDrive\Escritorio\PROYECTOS DE PYTHON"
    r"\quant-portfolio-project\data\model_results"
    r"\optimized_hyperparameters.json")

with open(json_path, "r", encoding="utf-8") as file:
    optimized_hyperparameters = json.load(file)

# -----------------------------------------------------------------------------
# Add Random Forest
# -----------------------------------------------------------------------------

optimized_hyperparameters["Random Forest"] = {
    "Percentile Rank": best_params_rf_rank
}

# -----------------------------------------------------------------------------
# Save updated JSON
# -----------------------------------------------------------------------------

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(
        optimized_hyperparameters,
        file,
        indent=4,
    )

print("JSON updated successfully.")

JSON updated successfully.


In [45]:
# -----------------------------------------------------------------------------
# Load existing hyperparameter table
# -----------------------------------------------------------------------------

optimized_hyperparameters_table = pd.read_parquet("../data/model_results/optimized_hyperparameters_table.parquet")

# -----------------------------------------------------------------------------
# Build Random Forest rows
# -----------------------------------------------------------------------------

rf_rows = []

for parameter, value in best_params_rf_rank.items():

    rf_rows.append({
        "Model": "Random Forest",
        "Scaling / Representation": "Percentile Rank",
        "Hyperparameter": parameter,
        "Optimal Value": value,
    })

rf_hyperparameters_table = pd.DataFrame(rf_rows)

# -----------------------------------------------------------------------------
# Append Random Forest
# -----------------------------------------------------------------------------

optimized_hyperparameters_table = pd.concat(
    [
        optimized_hyperparameters_table,
        rf_hyperparameters_table,
    ],
    ignore_index=True,
)

# -----------------------------------------------------------------------------
# Save updated table
# -----------------------------------------------------------------------------

optimized_hyperparameters_table.to_parquet("../data/model_results/optimized_hyperparameters_table.parquet")

print("Hyperparameter table updated successfully.")

Hyperparameter table updated successfully.


## 9. Comparativa Global de Resultados

### 9.1 Criterios de selección

La elección del modelo final no responde de manera aislada al valor más alto de Rank IC Mean, sino a una evaluación holística basada en el Score Multicriterio. Este marco integrador pondera de forma conjunta la capacidad pura de ordenación (Rank IC Mean), la consistencia temporal del rendimiento (Rank IC IR), la precisión en el signo de la predicción (Hit Rate) y la significancia estadística del α frente al ruido (t-statistic). 

Asimismo, el proceso de decisión examina el grado de respuesta y adaptabilidad de cada arquitectura tras la optimización de hiperparámetros con Optuna, contrastando de forma sistemática el desempeño de los algoritmos no lineales frente a la solidez de la referencia lineal (benchmark).

### 9.2 Modelo seleccionado

El modelo seleccionado como candidato definitivo es XGBoost bajo escalado Percentile Rank. Aunque Random Forest y Regresión Ridge conservan una leve ventaja en la fuerza bruta de ordenación (Rank IC Mean de 0.0337 y 0.0343 respectivamente, frente a los 0.0329 de XGBoost), XGBoost desplaza a la referencia lineal y al resto de arquitecturas de árboles para coronar el ranking global gracias a un perfil predictivo sensiblemente más equilibrado y robusto. La arquitectura de gradient boosting optimizada demuestra una clara superioridad en la estabilidad temporal del factor, alcanzando el mayor Information Ratio de la comparativa (0.2391, frente al 0.2079 de Random Forest y el 0.1828 de Ridge) y la mayor fiabilidad direccional de los modelos de cabeza (Hit Rate del 58.43%).

Esta consistencia se traduce en el $t$-statistic más elevado de todo el estudio (7.5623), confirmando que la señal generada por XGBoost tras el tuning presenta una significancia estadística sustancialmente superior y un menor riesgo de degradación fuera de muestra que sus competidores directos.

### 9.3 Configuración final

El comportamiento de **XGBoost (Percentile Rank)** ilustra el papel decisivo que desempeña la regularización en entornos de alta volatilidad y bajo ratio señal-ruido, propios de los retornos financieros *cross-sectional*. En su configuración por defecto, XGBoost registró el rendimiento más deficiente de todo el panel *baseline* (`Rank IC Mean` de **0.0122** y un *Score* de apenas **0.2250**), lo que sugería una aparente falta de competitividad para este problema. Sin embargo, tras la optimización con Optuna (20 *trials*, `TPESampler`), el modelo escaló hasta la primera posición del ranking global, multiplicando por casi tres su capacidad predictiva pura (`Rank IC Mean` de **0.0329**) y alcanzando los máximos del estudio en estabilidad (`IR` de **0.2391**) y significancia estadística ($t\text{-stat}$ de **7.5623**).

Analizando la parametrización óptima hallada (`learning_rate = 0.051`, `max_depth = 4`, `gamma = 4.47`, `reg_lambda = 4.87`, `min_child_weight = 9.73`), se observa un patrón claro de **regularización severa**. Aunque una profundidad máxima de 4 permite capturar interacciones moderadas, los elevados valores de `gamma` (que exige un umbral alto de reducción de pérdida para autorizar cada división) y de `min_child_weight` (que exige un peso significativo en las observaciones de cada hoja) fuerzan la poda efectiva de los árboles. En la práctica, estas restricciones impiden que la arquitectura construya particiones complejas o ruidosas sobre un conjunto de datos compuesto por solo tres *features*.

En consecuencia, el éxito de XGBoost tras la fase de *tuning* no obedece al aprovechamiento de relaciones altamente no lineales, sino a la simplificación estructural del ensemble. Al penalizar la complejidad, Optuna guió al algoritmo hacia un régimen de operación altamente regularizado que aproxima la monotonía y estabilidad del modelo lineal (Ridge), pero beneficiándose de la flexibilidad direccional implícita en la arquitectura de *gradient boosting*.


In [ ]:
print("\n" + "=" * 80)
print("FINAL MODEL CONFIGURATION")
print("=" * 80)

print("\nModel: XGBoost")
print("Representation: Percentile Rank")

print("\n" + "-" * 80)
print("OPTIMAL HYPERPARAMETERS")
print("-" * 80)

for parameter, value in best_params_xgb_rank.items():
    if isinstance(value, float):
        print(f"{parameter:<20} = {value:.6f}")
    else:
        print(f"{parameter:<20} = {value}")

print("\n" + "=" * 80)


FINAL MODEL CONFIGURATION

Model: XGBoost
Representation: Percentile Rank

--------------------------------------------------------------------------------
OPTIMAL HYPERPARAMETERS
--------------------------------------------------------------------------------
learning_rate        = 0.051438
max_depth            = 4
min_child_weight     = 9.726262
subsample            = 0.910053
colsample_bytree     = 0.975800
gamma                = 4.474137
reg_alpha            = 0.097611
reg_lambda           = 4.869641



### 9.4 Implicaciones para el backtesting

Identificar el modelo con mayor capacidad predictiva fuera de muestra es un paso fundamental, pero no garantiza por sí solo que la estrategia sea ejecutable o rentable en la práctica. Para verificar si métricas como el Rank IC o el Information Ratio se traducen en un rendimiento real, es imprescindible someter las predicciones a un backtesting que incorpore costes de transacción, turnover, drawdowns y el ratio de Sharpe final.

En lugar de evaluar únicamente al ganador, se seleccionan tres modelos representativos para analizar su comportamiento en cartera, incorporando a Random Forest en sustitución de LightGBM tras situarse en el segundo puesto del ranking global:

1. **XGBoost (Percentile Rank)**: El modelo ganador del ranking (Score 0.7167), que representa la hipótesis de un gradient boosting no lineal pero fuertemente regularizado.

2. **Random Forest (Percentile Rank)**: Segundo clasificado general (Score 0.6833) y máxima referencia del bagging. Supera a XGBoost en fuerza bruta de ordenación (Rank IC de 0.0337) y ofrece un equilibrio superior entre consistencia (IR de 0.2079) y capacidad de captura no lineal.

3. **Regresión Ridge (Percentile Rank)**: El benchmark lineal por excelencia (Score 0.6333). Ofrece la mayor fuerza de ordenación pura de todo el estudio (0.0343), siendo un modelo infinitamente más simple y estable. Permitirá comprobar si la complejidad no lineal de los ensembles basados en árboles aporta valor neto real tras deducir costes y rotación.

Se descartan la Regresión Lineal pura por ser idéntica a Ridge y LightGBM al quedar desplazado en el podio por el mejor desempeño relativo de Random Forest.

Con esta selección, el backtesting contrastará tres escenarios clave: si el mercado premia el gradient boosting regularizado (XGBoost), si la combinación de árboles vía bagging genera una mejor relación rentabilidad-riesgo en cartera (Random Forest), o si la señal real es esencialmente lineal y libre del coste computacional de los ensembles (Ridge).

## 10. Interpretabilidad y Diagnóstico de los Modelos Seleccionados

### 10.1 Importancia relativa de variables (Feature Importances)

En esta sección se examina el peso y la contribución relativa de las distintas variables predictoras en el panel de modelos seleccionados para la fase de backtesting: XGBoost (Percentile Rank), Random Forest (Percentile Rank) y Regresión Ridge (Percentile Rank). El objetivo es determinar la jerarquía interna de los factores dentro de cada arquitectura antes de analizar la naturaleza y dirección de sus efectos.

Para las arquitecturas basadas en ensembles de árboles (XGBoost y Random Forest), la importancia relativa se extrae directamente de las métricas nativas del algoritmo (basadas en la reducción de la impureza de Gini o el gain de las divisiones), lo que permite cuantificar el impacto estructural de cada variable en el proceso de partición del espacio de características.

Por su parte, en el modelo lineal (Regresión Ridge), la importancia relativa se deriva de la magnitud absoluta de sus coeficientes estandarizados. Al operar sobre la representación en Percentile Rank, las variables comparten un rango de escala idéntico, lo que permite comparar de forma directa e imparcial el peso específico de cada factor dentro del plano lineal.
    ├── Feature Importance
    └── Ranking de factores


In [ ]:
from src.models.evaluation import extract_cpcv_feature_importance

df_importance_ridge = extract_cpcv_feature_importance(
    model_cls=Ridge,
    model_params=best_params_ridge_rank,
    X=X_rank,
    y=y_rank,
    splits=splits,
)

In [ ]:
df_importance_xgb = extract_cpcv_feature_importance(
    model_cls=XGBRegressor,
    model_params=best_params_xgb_rank,
    X=X_rank,
    y=y_rank,
    splits=splits,
)

In [ ]:
df_importance_rf = extract_cpcv_feature_importance(
    model_cls=RandomForestRegressor,
    model_params=best_params_rf_rank,
    X=X_rank,
    y=y_rank,
    splits=splits,
)

In [ ]:
mean_importance_ridge = (
    df_importance_ridge
    .drop(columns="fold")
    .mean()
    .sort_values(ascending=False)
)

mean_importance_xgb = (
    df_importance_xgb
    .drop(columns="fold")
    .mean()
    .sort_values(ascending=False)
)

mean_importance_rf = (
    df_importance_rf
    .drop(columns="fold")
    .mean()
    .sort_values(ascending=False)
)

### 10.2 Análisis SHAP

El análisis SHAP permitirá complementar las medidas tradicionales de importancia proporcionando una interpretación más detallada de las predicciones de los modelos basados en árboles. A diferencia de una medida agregada de importancia, los valores SHAP permiten estudiar tanto la magnitud como la dirección de la contribución de cada factor a las predicciones individuales. 

Se utilizarán los Summary Plots y los gráficos de dependencia para analizar cómo los distintos niveles de cada factor afectan al retorno esperado y para detectar posibles relaciones no lineales, umbrales o efectos de saturación.

    ├── SHAP Summary Plot
    ├── SHAP Dependence — Momentum
    ├── SHAP Dependence — Upside Volatility
    └── SHAP Dependence — Amihud

### 10.3 Relevancia e interpretación económica

Finalmente, los resultados de importancia y SHAP se interpretarán desde una perspectiva económica y financiera. El objetivo será determinar si las relaciones aprendidas por los modelos son coherentes con el comportamiento esperado de los factores estudiados y comparar estos resultados con la evidencia obtenida durante el análisis factorial previo. 

Se prestará especial atención a la dirección de los efectos, a la posible existencia de relaciones no lineales y a las diferencias entre la importancia individual de los factores y su contribución dentro de un modelo multivariable. De este modo, la interpretabilidad servirá también como una comprobación de coherencia económica antes de evaluar el comportamiento de las estrategias en el backtesting.

    ├── Interpretación individual de cada factor
    ├── Comparación con análisis factorial previo
    ├── Relaciones lineales vs. no lineales
    └── Coherencia económica del modelo

## 11. Exportación de Artefactos y Resultados

### 11.1 Guardado en disco del Modelo Ganador (joblib/pickle)

### 11.2 Exportación de predicciones Out-of-Sample

### 11.3 Registro de metadata de entrenamiento

## 12. Conclusiones y Próximos Pasos